# Ozone and temperature in 2025

When did elevated ozone last for several hours, what temperatures accompanied it, and how much of the year did we observe? This notebook prepares the data for that analysis. Bronze preserves the original files. Silver cleans the observations and matches Alabama ozone readings with nearby weather reports. Gold uses the published hourly data to identify periods of elevated ozone.

We use [EPA AirData](https://aqs.epa.gov/aqsweb/airdata/FileFormats.html) for ozone and [NOAA GHCNh](https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/doc/ghcnh_DOCUMENTATION.pdf) for temperature.

The main flow is **save the source files → clean the records → match by hour → find ozone periods → publish the results**. We run it for Alabama. The functions at the end let us check Georgia reuse and record a Spark timing baseline when called separately.

## 1. Setup

On the VM, select the `spark-env` kernel. It uses Python 3.11 and the PySpark 3.5.0 libraries bundled with `/opt/spark`. The default environment is `dev`. If a source has several saved versions, select one by checksum in `source_versions`. Set `PIPELINE_SHOW_DIAGNOSTICS=true` to see extra sample rows.

The terminal runner has three modes. **Full** runs Bronze, Silver and Gold. **Prepare** stops after the hourly Silver receipt in 9.4d. **Analyse** starts a fresh process, opens the hourly Silver release named in a receipt and runs Gold. Running this notebook from top to bottom follows the full flow.

Most cells first define functions; the later execution cells call them. Spark transformations describe the work, while actions such as `count`, `collect`, `show` and writing Parquet actually run it.

In [1]:
# Load the Python tools used for files, run records and data processing.
import calendar
import csv
import hashlib
import json
import math
import os
import shutil
import subprocess
import sys
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from urllib.request import Request, urlopen
from urllib.parse import urlsplit
from zipfile import ZipFile

# Find the project whether Jupyter starts here or in the notebooks folder.
for folder in (Path.cwd(), *Path.cwd().parents):
    project_dir = folder if folder.name == "final_project" else folder / "final_project"
    if (project_dir / "config" / "sources.json").is_file():
        project_dir = project_dir.resolve()
        break
else:
    raise FileNotFoundError("Start Jupyter from inside the project checkout.")

# Make the helper files in this checkout available to the notebook.
sys.path.insert(0, str(project_dir / "src"))
from run_context import create_run_context
from source_columns import required_columns, check_columns
from silver_files import file_sha256, silver_json_sha256, silver_child_path, verify_silver_file_inventory

# Read the run settings supplied by the runner or the notebook environment.
target = os.environ.get("PIPELINE_TARGET", "vm")
environment = os.environ.get("PIPELINE_ENVIRONMENT", "dev")
if target != "vm":
    raise RuntimeError("Run this notebook on the VM with PIPELINE_TARGET=vm.")
# Reuse saved sources by default. An explicit refresh asks the provider for a new copy.
refresh_setting = os.environ.get(
    "PIPELINE_REFRESH_SOURCES", "false"
).lower()

if refresh_setting not in {"true", "false"}:
    raise ValueError("PIPELINE_REFRESH_SOURCES must be true or false.")

refresh_sources = refresh_setting == "true"
# Select the VM storage configuration and read the list of EPA and NOAA inputs.
environments_file = project_dir / "config" / "environments.json"
sources = json.loads((project_dir / "config/sources.json").read_text(encoding="utf-8"))

# Fill this in only if a source has more than one saved version.
source_versions = {}
# Example: source_versions["epa/hourly_ozone_2025"] = "<original SHA-256>"

# Check that this kernel uses spark-env, then load PySpark from /opt/spark.
from spark_runtime import configure_vm_python, start_silver_session
configure_vm_python()

import pyspark
from pyspark.sql import Window, functions as F

# Display the runtime so we can see which Python and Spark installation we are using.
print(f"Target: {target}; environment: {environment}; refresh: {refresh_sources}")
print("Python:", sys.executable, sys.version.split()[0])
print("PySpark:", pyspark.__version__)
print("PySpark location:", pyspark.__file__)

# Enable detailed sample tables when inspecting source results.
diagnostic_setting = os.environ.get("PIPELINE_SHOW_DIAGNOSTICS", "false").lower()
if diagnostic_setting not in {"true", "false"}:
    raise ValueError("PIPELINE_SHOW_DIAGNOSTICS must be true or false.")
show_diagnostics = diagnostic_setting == "true"

# Reuse the hourly transformations and verified release loaders from src.
from silver_hourly import (
    build_yearly_mapping, enrich_hourly_ozone, inspect_hourly_inputs,
    prepare_hourly_silver, select_hourly_temperature,
)
from silver_io import (
    check_hourly_evidence, hourly_expected_rows, hourly_output_keys,
    load_hourly_silver, load_source_silver, read_hourly_files,
)

Target: vm; environment: dev; refresh: False
Python: /home/ubuntu/spark-env/bin/python 3.11.15
PySpark: 3.5.0
PySpark location: /opt/spark/python/lib/pyspark.zip/pyspark/__init__.py


### Start Spark

We use UTC throughout so that EPA and NOAA timestamps refer to the same clock hour. A small sum checks that the session can run a job on the VM's Spark master.

In [2]:
# Create the run ID and storage paths shared by Silver and Gold.
silver_context = create_run_context(
    environments_file,
    environment,
    test_run_id=os.environ.get("PIPELINE_TEST_RUN_ID"),
)

# Start the shared Silver/Gold session in UTC and run the small startup check.
spark = start_silver_session(project_dir, silver_context)
print("Spark engine:", spark.version)
print("Master:", spark.sparkContext.master)
print("Session timezone:", spark.conf.get("spark.sql.session.timeZone"))
print("Silver environment root:", silver_context.environment_root)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/13 23:12:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark engine: 3.5.0
Master: spark://sparkvm-test:7077
Session timezone: UTC
Silver environment root: /home/ubuntu/dat535-final-project-data/dev


## 2. Bronze: preserve the source files

We keep the original EPA and NOAA files so another run can use the same inputs.

### 2.1 Record each file

A manifest records where a file came from, its size and its SHA-256 checksum. The checksum lets us check that a saved copy still has the same contents.

Bronze has its own acquisition report. The Silver and Gold sections share the run context created in Setup, so their report directory can differ from the Bronze directory. The manifests link the stages to the exact input files.

In [3]:
# Define small helpers for writing reports and describing source files.
def write_json(path, value):
    """Save to a temporary file first, then replace the JSON file."""
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2) + "\n", encoding="utf-8")
    temporary.replace(path)


def source_name(source):
    """Return a dataset name including its year and station when supplied."""
    name = source["dataset"]
    for field in ("year", "station_id"):
        if field in source:
            name += "_" + str(source[field])
    return name





def file_record(path, checksum, root):
    """Return the file path relative to the environment root, its size and supplied checksum."""
    return {
        "path": path.relative_to(root).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": checksum,
    }

### Record the run settings

We save the settings used for this run. When we run from a Git checkout, we also record the commit and whether there are uncommitted changes.

In [4]:
# Record which settings and code produced this run.
def capture_provenance(context):
    """Save effective settings and return the code and configuration identity."""
    # Save the settings actually used, then hash the file so changes can be detected.
    config_path = context.reports_dir / "effective_config.json"
    write_json(config_path, {
        "sources": sources, "target": target, "environment": context.environment,
        "environment_root": str(context.environment_root),
        "refresh_sources": refresh_sources, "source_versions": source_versions,
    })

    # A copied VM folder has no Git history to read, so these values stay unknown.
    provenance = {
        "code_source": "manual_copy",
        "code_revision": None,
        "working_tree_dirty": None,
        "config_sha256": file_sha256(config_path),
        "evidence_path": context.reports_dir.relative_to(context.environment_root).as_posix(),
    }
    # Read Git details when this folder belongs to a checkout.
    in_git_checkout = any(
        (folder / ".git").exists() for folder in (project_dir, *project_dir.parents)
    )
    if in_git_checkout:
        changes = subprocess.check_output(
            ["git", "status", "--porcelain"], cwd=project_dir, text=True
        ).strip()
        provenance.update({
            "code_source": "git",
            "code_revision": subprocess.check_output(
                ["git", "rev-parse", "HEAD"], cwd=project_dir, text=True
            ).strip(),
            "working_tree_dirty": bool(changes),
        })
    # Return these details for the source manifests and later run reports.
    return provenance


### 2.2 Download and extract files

We keep each EPA ZIP alongside its extracted CSV. NOAA files can be read directly. Before reusing a saved file, we check its size and checksum against the manifest.

In [5]:
# Define the download, extraction and saved-file checks used by Bronze.
def download_file(url, destination):
    """Download a complete file and return its URLs and retrieval time."""
    destination.parent.mkdir(parents=True, exist_ok=True)
    # Use a .part file so an interrupted download cannot look like a completed input.
    partial = destination.with_name(destination.name + ".part")
    # Try up to three times; a failed attempt must not become the completed file.
    for attempt in range(1, 4):
        try:
            request = Request(url, headers={"Accept-Encoding": "identity"})
            with urlopen(request, timeout=60) as response:
                expected_size = response.headers.get("Content-Length")
                resolved_url = response.geturl()
                with partial.open("wb") as handle:
                    shutil.copyfileobj(response, handle, length=1024 * 1024)
            # Reject empty or incomplete downloads before renaming the partial file.
            size = partial.stat().st_size
            if size == 0 or (expected_size is not None and size != int(expected_size)):
                raise ValueError("Download is empty or has an unexpected size.")
            partial.rename(destination)
            return {
                "requested_url": url, "resolved_url": resolved_url,
                "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
            }
        except (OSError, ValueError) as error:
            if attempt == 3:
                raise
            print(f"Retry {attempt}: {error}")
            time.sleep(2)


def read_header(path):
    """Return the column names from a CSV or pipe-delimited source."""
    # NOAA uses pipes; the EPA CSV files use commas.
    delimiter = "|" if path.suffix == ".psv" else ","
    with path.open(newline="", encoding="utf-8-sig") as handle:
        return next(csv.reader(handle, delimiter=delimiter), [])


def prepare_file(path):
    """Extract the CSV from an EPA ZIP, or return the plain NOAA file."""
    if path.suffix != ".zip":
        return path
    csv_path = path.with_suffix(".csv")
    with ZipFile(path) as archive:
        with archive.open(csv_path.name) as source, csv_path.open("xb") as destination:
            shutil.copyfileobj(source, destination)
    return csv_path


def stored_data_path(manifest, context):
    """Verify saved file sizes and checksums, then return the file Silver should read."""
    # Verify both the original download and any extracted files against the manifest.
    for record in [manifest["original_file"], *manifest["extracted_files"]]:
        path = context.environment_root / record["path"]
        if not path.is_file() or path.stat().st_size != record["size_bytes"]:
            raise ValueError(f"Stored file is missing or its size changed: {path}")
        if file_sha256(path) != record["sha256"]:
            raise ValueError(f"Stored file checksum differs: {path}")
    # Return the extracted CSV when present, otherwise the original data file.
    if manifest["extracted_files"]:
        data_file = manifest["extracted_files"][0]
    else:
        data_file = manifest["original_file"]
    return context.environment_root / data_file["path"]

### 2.3 Select a source version

We reuse a verified copy when one is available. A complete file with missing required columns can still be saved in Bronze, but Silver will reject it.

In [6]:
# Select one saved source version or download and record a source snapshot.
def acquire_source(source, context, provenance):
    """Find a saved source version or download it, then return its details."""
    # Build the source identity and the directory containing its version manifests.
    name = source_name(source)
    key = source["source"] + "/" + name
    published = context.published_root / "bronze" / source["source"] / name
    version = source_versions.get(key)
    manifest_path = None
    identity = {"provider": source["source"], "dataset": source["dataset"]}
    for field in ("year", "station_id"):
        if field in source:
            identity[field] = source[field]

    # Without refresh, use the requested checksum or the single saved version.
    if not refresh_sources:
        if version is not None:
            manifest_path = published / version / "manifest.json"
            if not manifest_path.is_file():
                raise FileNotFoundError(
                    f"Selected source version is missing: {manifest_path}"
                )
        else:
            candidates = sorted(published.glob("*/manifest.json"))
            # Require an explicit choice when several versions exist; do not guess the newest.
            if len(candidates) > 1:
                raise ValueError(
                    f"Select a checksum in source_versions for {key}."
                )
            manifest_path = candidates[0] if candidates else None

    # Download when refresh is requested or no saved version was found.
    # An explicitly requested version that is missing has already raised an error.
    downloaded = manifest_path is None
    acquisition = None
    # Reuse the saved file only after its size and checksum have been verified.
    if not downloaded:
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        data_path = stored_data_path(
            manifest, context
        )
        outcome = "reused"
    else:
        # Download into this run's staging directory, then inspect the data header.
        staging = context.staging_root / "bronze" / source["source"] / name
        filename = Path(urlsplit(source["url"]).path).name
        original = staging / filename
        acquisition = download_file(source["url"], original)
        acquisition["run_id"] = context.run_id
        data = prepare_file(original)
        schema = check_columns(source, read_header(data))
        # Use the original file contents to identify the source version.
        checksum = file_sha256(original)
        data_checksum = checksum if data == original else file_sha256(data)
        snapshot = (
            context.data_root / "bronze" / source["source"] / name / checksum
        )
        # Check whether a snapshot with this checksum already exists; verify it below before reuse.
        reused = snapshot.exists()
        if not reused:
            snapshot.parent.mkdir(parents=True, exist_ok=True)
            staging.rename(snapshot)

        # The manifest points to the saved files, not the staging folder.
        original_record = file_record(
            snapshot / original.name,
            checksum,
            context.environment_root,
        )
        extracted = []
        if data != original:
            extracted_record = file_record(
                snapshot / data.name,
                data_checksum,
                context.environment_root,
            )
            extracted_record["archive_member"] = data.name
            extracted.append(extracted_record)

        # Describe the files, their structure and how this download was obtained.
        candidate = {
            "manifest_version": 1,
            "source": identity,
            "acquisition": acquisition,
            "original_file": original_record,
            "extracted_files": extracted,
            "schema": schema,
            "provenance": provenance,
        }
        if reused:
            stored_data_path(candidate, context)

        manifest_path = published / checksum / "manifest.json"
        if manifest_path.exists():
            # Preserve the original acquisition details.
            manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        else:
            manifest = candidate
            write_json(manifest_path, manifest)

        data_path = snapshot / data.name
        outcome = "reused" if reused else "acquired"

        # Remove this run's extra copies after verifying the saved version.
        if reused:
            if not staging.resolve().is_relative_to(
                context.staging_root.resolve()
            ):
                raise ValueError(
                    "Duplicate cleanup must stay inside this run's staging."
                )
            for path in {original, data}:
                path.unlink()
            if not any(staging.iterdir()):
                staging.rmdir()

    # Return the paths and schema status needed by the later Silver readers.
    return {
        "request": source,
        "manifest_path": manifest_path,
        "data_path": data_path,
        "schema": manifest["schema"],
        "outcome": outcome,
        "downloaded": downloaded,
        "acquisition": acquisition,
    }

### 2.4 Record acquisition results

Each environment has a lock so two runs cannot write its Bronze files at the same time. The run report records the outcome for each source.

In [7]:
# Run acquisition for all configured sources and record success or failure.
def run_bronze():
    """Acquire the configured inputs, save a run report and return their file details."""
    # Bronze has its own run directory. Silver and Gold share the context created in Setup.
    context = create_run_context(
        environments_file, environment,
        test_run_id=os.environ.get("PIPELINE_TEST_RUN_ID"),
    )
    context.reports_dir.mkdir(parents=True, exist_ok=True)
    started = time.perf_counter()
    # Mark success only after every source was acquired or verified for reuse.
    report = {
        "run_id": context.run_id, "environment": context.environment,
        "status": "failure", "started_at_utc": datetime.now(timezone.utc).isoformat(),
        "refresh_sources": refresh_sources, "results": [], "blocked_sources": [],
    }
    downloads = []
    lock = context.environment_root / ".bronze-publication.lock"
    locked = False
    print(f"Target: {target}; environment: {environment}; refresh: {refresh_sources}")
    print(f"Run: {context.run_id}")
    try:
        provenance = capture_provenance(context)
        report["provenance"] = provenance
        # Allow only one acquisition run to write to this environment at a time.
        lock.mkdir()
        locked = True
        # Acquire each source and keep its selected file details for downstream processing.
        for source in sources:
            try:
                item = acquire_source(source, context, provenance)
            except Exception as error:
                report["results"].append({
                    "request": source, "outcome": "failed", "error": str(error),
                })
                raise
            downloads.append(item)
            report["results"].append({
                "request": source, "outcome": item["outcome"],
                "downloaded": item["downloaded"],
                "manifest_path": item["manifest_path"].relative_to(
                    context.environment_root
                ).as_posix(),
                "acquisition": item["acquisition"], "error": None,
            })
            # Bronze can preserve a file with an unusable header; Silver must reject it.
            if item["schema"]["status"] == "blocked":
                report["blocked_sources"].append(source)
            action = "Downloaded" if item["downloaded"] else "Reused"
            print(f"{action}: {source['source']}/{source_name(source)} "
                  f"| {item['schema']['status']} | {item['data_path']}")
        report["status"] = "success"
    except Exception as error:
        report["error"] = str(error)
        raise
    finally:
        # Save a report even if one of the sources failed.
        if locked:
            lock.rmdir()
        report["finished_at_utc"] = datetime.now(timezone.utc).isoformat()
        report["elapsed_seconds"] = round(time.perf_counter() - started, 3)
        report_path = context.reports_dir / "run.json"
        write_json(report_path, report)
        print(f"Run report: {report_path}")
    # Return file paths and source details; Silver reads the actual records later.
    return downloads

### 2.5 Load the selected inputs

This run reuses saved files and downloads any missing inputs.

In [8]:
# Execute Bronze now and keep the source selections for the Silver sections.
downloads = run_bronze()

Target: vm; environment: dev; refresh: False
Run: 7dcc89d90fbc4ef7ba5c7ef8f4e9bed8
Reused: epa/sites | compatible | /home/ubuntu/dat535-final-project-data/dev/data/bronze/epa/sites/56b322f6a3d5368d055bce53d0988275c0b27fbcced3d57a6ce8f2e2b539e15a/aqs_sites.csv
Reused: epa/monitors | compatible | /home/ubuntu/dat535-final-project-data/dev/data/bronze/epa/monitors/e8325216d4ec5df462a23c201e6ffa1176b2fce363c9f6b0e3f18ce6ad958525/aqs_monitors.csv
Reused: epa/hourly_ozone_2025 | compatible | /home/ubuntu/dat535-final-project-data/dev/data/bronze/epa/hourly_ozone_2025/2b7b9e03b43a2bf20d3c3e78fc8c98b6b349fe2c3c4778382a2997144e7b2fd7/hourly_44201_2025.csv
Reused: noaa/stations | compatible | /home/ubuntu/dat535-final-project-data/dev/data/bronze/noaa/stations/812ea50bd5efd1b9cc09d5800c55825b3d43b3c5aa75ab17b2848f43b1bfaaef/ghcnh-station-list.csv
Reused: noaa/hourly_weather_2025_USW00053864 | compatible | /home/ubuntu/dat535-final-project-data/dev/data/bronze/noaa/hourly_weather_2025_USW00053864

## 3. Alabama profile

We pair the McAdory ozone monitor with the Alabaster weather station, about 26.7 km away. The profile gives us their identifiers. We assess the pairing in section 9.3 once we know how many hours have usable observations.

In [9]:
# Load the Alabama monitor, weather station and year used by the main pipeline.
alabama_profile = json.loads(
    (project_dir / "config" / "alabama_profile.json").read_text(encoding="utf-8")
)


## 4. Georgia profile

The South DeKalb monitor and Atlanta weather station use the same profile format as Alabama. The coordinate reference system used by the EPA records still needs review before we can approve this pairing.

In [10]:
# Load the Georgia settings for the later reuse check; this does not run or approve it.
georgia_profile = json.loads(
    (project_dir / "config" / "georgia_profile.json").read_text(encoding="utf-8")
)


## 5. Silver: ozone observations

### 5.1 Read the source records

We check the header, quoting and number of fields before Spark reads the data. We read the fields as strings first to preserve identifiers such as Alabama's state code, `01`.

In the Silver sections, **eligible** means a record passes our analysis rules. A **quality flag** describes a retained record that needs context. The **audit** accounts for accepted records, exclusions and extra copies. **Lineage** is the source-file and original-row information that lets us trace a result back to Bronze.

In [ ]:
# Check the file structure before reading its fields into Spark as text.
# Return the DataFrame together with the column names and structural row count.
def read_validated_delimited_file(path, delimiter):
    """Return a string-column DataFrame plus the checked header and source row count."""
    path = Path(path).resolve(strict=True)
    record_count = 0

    # Scan with Python first so malformed rows fail before Spark processing begins.
    with path.open(encoding="utf-8-sig", newline="") as handle:
        reader = csv.reader(
            handle,
            delimiter=delimiter,
            quotechar='"',
            doublequote=True,
            escapechar=None,
            skipinitialspace=False,
            strict=True,
        )

        # Check the column names and each row's structure, and count the data rows.
        # Stop with an error if the file does not match the expected format.
        try:
            columns = next(reader, None)

            if not columns or any(
                not name or name != name.strip()
                for name in columns
            ):
                raise ValueError(
                    f"{path.name}: missing, blank or untrimmed column names"
                )

            if len({name.casefold() for name in columns}) != len(columns):
                raise ValueError(f"{path.name}: duplicate column names")

            if reader.line_num != 1:
                raise ValueError(
                    f"{path.name}: multiline headers are unsupported"
                )

            previous_line = reader.line_num

            for values in reader:
                if reader.line_num != previous_line + 1:
                    raise ValueError(
                        f"{path.name}: multiline record ending at "
                        f"line {reader.line_num} is unsupported"
                    )

                if len(values) != len(columns):
                    raise ValueError(
                        f"{path.name}: line {reader.line_num}: "
                        f"expected {len(columns)} fields, "
                        f"found {len(values)}"
                    )

                record_count += 1
                previous_line = reader.line_num

        except csv.Error as error:
            raise ValueError(
                f"{path.name}: CSV parsing failed near "
                f"line {reader.line_num}: {error}"
            ) from error

    # Keep all fields as text so identifiers such as state code 01 retain leading zeros.
    schema = pyspark.sql.types.StructType()
    for name in columns:
        schema.add(name, "string", nullable=True)

    # Read the checked file into Spark with the text schema and strict parsing.
    records = (
        spark.read
        .schema(schema)
        .option("header", True)
        .option("enforceSchema", False)
        .option("encoding", "UTF-8")
        .option("sep", delimiter)
        .option("quote", '"')
        .option("escape", '"')
        .option("unescapedQuoteHandling", "RAISE_ERROR")
        .option("multiLine", False)
        .option("ignoreLeadingWhiteSpace", False)
        .option("ignoreTrailingWhiteSpace", False)
        .option("mode", "FAILFAST")
        .csv(str(path))
    )

    return records, {
        "columns": columns,
        "records": record_count,
    }


def read_validated_ozone_csv(path):
    """Read ozone CSV using the shared structural checks."""
    return read_validated_delimited_file(path, delimiter=",")


# Build a Spark condition for missing text; it is applied to rows later.
def missing_string(column_name):
    """Identify null, empty or space-only source values."""
    value = F.col(column_name)
    return value.isNull() | (F.trim(value) == "")


def finite_number(column_name):
    """Return a Spark condition excluding null, NaN and positive or negative infinity."""
    value = F.col(column_name)
    return (
        value.isNotNull()
        & ~F.isnan(value)
        & (F.abs(value) != float("inf"))
    )


### 5.2 Apply the ozone quality rules

This section defines how we check ozone records and explain whether they are usable.

1. **Convert the values.** Combine the source date and time into a UTC timestamp, and convert the ozone reading and method detection limit (MDL) into numbers. Keep the original text so we can distinguish missing values from values that could not be converted.

2. **Check for problems.** Check the required identifiers, timestamps, units, measurements, and source qualifier codes. These checks help us identify records that do not meet the needs of our hourly analysis.

3. **Record the outcome.** Mark whether each row passes the rules and keep every failure reason. Also keep one summary reason: the first failed rule in the list. This gives us a short label while preserving the full explanation.

Our rules exclude qualifier `6` and unsupported qualifier codes. Small negative ozone readings down to −0.004 ppm are allowed and flagged. A missing or unusable MDL does not, by itself, exclude an ozone reading.

No rows are removed here. The results are used later to build the clean observation table and the audit explaining what happened to the source records.

In [ ]:
# Convert dates and measurements from text, keeping the original columns.
# Values that cannot be parsed become null for the later checks.
def type_ozone_fields(records):
    """Add typed fields while retaining the source columns."""
    if records.sparkSession.conf.get("spark.sql.session.timeZone") != "UTC":
        raise ValueError("Ozone timestamp parsing requires a UTC Spark session.")

    # Combine the EPA GMT date and time, then parse them using the UTC Spark session.
    sample_time = F.concat(
        F.col("Date GMT"), F.lit(" "), F.col("Time GMT")
    )

    return (
        records
        .withColumn(
            "timestamp_utc",
            F.try_to_timestamp(sample_time, F.lit("yyyy-MM-dd HH:mm")),
        )
        .withColumn(
            "ozone_value",
            F.expr("try_cast(`Sample Measurement` AS DOUBLE)"),
        )
        .withColumn(
            "mdl_value",
            F.expr("try_cast(`MDL` AS DOUBLE)"),
        )
    )

# Check for missing required identifiers: state, county, site, parameter, POC. 
# These Spark conditions describe checks; the processing function applies them to rows.
missing_identifier = F.lit(False)
for column_name in (
    "State Code", "County Code", "Site Num", "Parameter Code", "POC"
):
    missing_identifier = missing_identifier | missing_string(column_name)

# Check for missing date, time, or measurement units.
missing_time = missing_string("Date GMT") | missing_string("Time GMT")
missing_units = missing_string("Units of Measure")

# Collect the checks for identifiers, timestamps, and units.
ozone_field_checks = {
    "missing_identifier": missing_identifier,
    "missing_timestamp": missing_time,
    "invalid_timestamp": ~missing_time & F.col("timestamp_utc").isNull(),
    "missing_units": missing_units,
    "unsupported_units": (
        ~missing_units
        & (F.col("Units of Measure") != "Parts per million")
    ),
}

# Check ozone and MDL for missing, invalid, non-finite, or negative values.
for label, source_column, typed_column in (
    ("ozone", "Sample Measurement", "ozone_value"),
    ("mdl", "MDL", "mdl_value"),
):
    # Check for missing text and a finite numeric value.
    missing = missing_string(source_column)
    value = F.col(typed_column)
    finite = finite_number(typed_column)

    # Build the same diagnostics for ozone and its method detection limit (MDL).
    ozone_field_checks.update({
        f"missing_{label}": missing,
        f"invalid_{label}": ~missing & value.isNull(),
        f"nonfinite_{label}": value.isNotNull() & ~finite,
        f"negative_{label}": finite & (value < 0),
    })


# Mark which rows pass the quality rules and keep the reasons for failures.
def classify_ozone_fields(records, year):
    """Classify checked ozone records using ordered field rules."""
    # Normalise qualifier text before comparing it with the supported codes.
    allowed_qualifiers = ["", "XS", "MD", "1V", "V"]
    qualifier = F.coalesce(F.trim(F.col("Qualifier")), F.lit(""))
    timestamp = F.col("timestamp_utc")
    value = F.col("ozone_value")

    # Keep all failures, but use the first in this list as the single audit outcome.
    rules = [
        ("missing_identifier", F.col("missing_identifier")),
        ("unusable_timestamp", timestamp.isNull()),
        (
            "outside_year",
            timestamp.isNotNull() & (F.year(timestamp) != year),
        ),
        (
            "off_hour",
            timestamp.isNotNull()
            & (timestamp != F.date_trunc("hour", timestamp)),
        ),
        (
            "unusable_units",
            missing_string("Units of Measure")
            | (F.col("Units of Measure") != "Parts per million"),
        ),
        ("unusable_ozone", ~finite_number("ozone_value")),
        (
            "below_minimum",
            finite_number("ozone_value") & (value < -0.004),
        ),
        ("qualifier_6", qualifier == "6"),
        (
            "unsupported_qualifier",
            ~qualifier.isin(allowed_qualifiers + ["6"]),
        ),
    ]

    # Collect all failure reasons in rule order, leaving out rules that passed.
    reasons = F.filter(
        F.array(*[
            F.when(condition, F.lit(reason))
            for reason, condition in rules
        ]),
        lambda reason: reason.isNotNull(),
    )

    # Add the normalised qualifier, all failure reasons and one summary outcome.
    return (
        records
        .withColumn("qualifier_code", qualifier)
        .withColumn("field_reasons", reasons)
        .withColumn("field_eligible", F.size("field_reasons") == 0)
        .withColumn(
            "field_disposition",
            F.when(F.col("field_eligible"), F.lit("eligible"))
            .otherwise(F.element_at("field_reasons", 1)),
        )
    )


### 5.3 Resolve duplicate readings

We expect one ozone observation per monitor and UTC timestamp. Together, the monitor identifiers and timestamp form the observation's key.

This section defines how we trace records to their source and handle repeated records:

1. **Record where each row came from.** Attach the Bronze source file, its version, and a fingerprint of the row's original values. This lets us trace unexpected results back to the original input.

2. **Combine identical copies.** Group rows that match in every column and keep one row per group. Record how many copies there were so we avoid counting the same observation repeatedly while still accounting for every input row.

3. **Flag conflicting records.** Within the same source file and version, different records sharing the same usable key are marked as conflicting and ineligible. We keep the conflict information rather than arbitrarily choosing one record.

The result is one DataFrame containing both eligible and problem records, with copy counts, conflict flags, and eligibility decisions. Conflicting records remain available for the audit.

The two functions here are called by `process_ozone_source` in **5.7**. It then uses the result to build the observation table with the function from **5.4** and the audit with the function from **5.5**.

In [ ]:
# Together these identifiers and the UTC timestamp identify one ozone observation.
ozone_key_columns = [
    "State Code",
    "County Code",
    "Site Num",
    "Parameter Code",
    "POC",
    "timestamp_utc",
]

# Add source file details and a fingerprint of each row's original values.
# Keep this link so unexpected results can be checked against the original input.
def attach_bronze_lineage(records, source, environment_root, source_columns):
    """Add the source file details and a hash of each original row for tracing results."""

    # Get full paths and read the Bronze manifest.
    root = Path(environment_root).resolve()
    manifest_path = Path(source["manifest_path"]).resolve()
    data_path = Path(source["data_path"]).resolve()
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

    if manifest["manifest_version"] != 1:
        raise ValueError("Unsupported Bronze manifest version.")

    # Find the manifest entry for the data file we are reading.
    relative_data_path = data_path.relative_to(root).as_posix()
    file_entries = [
        manifest["original_file"],
        *manifest["extracted_files"],
    ]
    matching_files = [
        entry for entry in file_entries
        if entry["path"] == relative_data_path
    ]

    # Stop unless the data file matches exactly one manifest entry.
    if len(matching_files) != 1:
        raise ValueError("Selected data file must match one manifest entry.")

    # Check that the manifest folder name matches the recorded source checksum.
    source_version = manifest["original_file"]["sha256"]
    if manifest_path.parent.name != source_version:
        raise ValueError("Manifest directory does not match its source version.")

    # Collect source details to add to each row.
    metadata = {
        "source_provider": manifest["source"]["provider"],
        "source_dataset": manifest["source"]["dataset"],
        "source_version_sha256": source_version,
        "source_file_path": relative_data_path,
        "source_file_sha256": matching_files[0]["sha256"],
        "source_manifest_path": manifest_path.relative_to(root).as_posix(),
        "source_record_hash_scheme": "sha256-json-raw-fields-v1",
    }

    # Hash the original values, including nulls, in a fixed field order.
    # This lets the audit and clean output refer to the same original row.
    source_record_json = F.to_json(
        F.struct(*[F.col(name) for name in sorted(source_columns)]),
        options={"ignoreNullFields": "false"},
    )

    # Return all columns with source details and a fingerprint for each row.
    return records.select(
        "*",
        *[F.lit(value).alias(name) for name, value in metadata.items()],
        F.sha2(source_record_json, 256).alias("source_record_sha256"),
    )


# Keep one copy of identical rows and record their count.
# Mark conflicting records with the same usable key as ineligible.
def resolve_source_keys(records, key_columns, key_is_usable):
    """Count identical copies and mark different rows sharing a usable key as ineligible."""

    # Keep conflict checks within the same source file and version.
    source_scope = [
        "source_provider",
        "source_dataset",
        "source_version_sha256",
        "source_file_path",
        "source_file_sha256",
    ]

    # Group rows that match in every column and count their copies.
    distinct_records = records.groupBy(*records.columns).agg(
        F.count("*").alias("source_occurrences")
    )

    # The window groups matching source/key rows without collapsing them.
    # After exact copies are grouped, more than one row means conflicting evidence.
    key_window = Window.partitionBy(*(source_scope + key_columns))
    usable_key = F.coalesce(key_is_usable, F.lit(False))

    # Return rows with duplicate counts, conflict flags, and eligibility.
    return (
        distinct_records
        .withColumn(
            "duplicate_extra_records",
            F.col("source_occurrences") - 1,
        )
        .withColumn(
            "key_variant_count",
            F.when(
                usable_key,
                F.count(F.lit(1)).over(key_window),
            ),
        )
        .withColumn(
            "key_conflict",
            F.coalesce(F.col("key_variant_count") > 1, F.lit(False)),
        )
        .withColumn(
            "record_disposition",
            F.when(F.col("key_conflict"), F.lit("conflicting_key"))
            .otherwise(F.col("field_disposition")),
        )
        .withColumn(
            "record_reasons",
            F.when(
                F.col("key_conflict"),
                F.concat(
                    F.array(F.lit("conflicting_key")),
                    F.col("field_reasons"),
                ),
            ).otherwise(F.col("field_reasons")),
        )
        .withColumn(
            "record_eligible",
            F.col("record_disposition") == "eligible",
        )
    )


### 5.4 Build the ozone observation table

This section defines how we turn the checked records into the ozone observation table used for later analysis.

1. **Keep eligible readings.** Use the decisions from the earlier quality and duplicate checks to select the rows that can be used.

2. **Prepare the output columns.** Rename the identifier columns, combine them into a monitor ID, and keep the timestamp and measured ozone value. Keep the MDL only when it is finite and at least zero; otherwise, store it as null.

3. **Keep useful context.** Add quality flags for conditions such as small negative readings, values below the detection limit, and source qualifiers. These flags describe readings we retain; they are not additional exclusion rules.

We also keep the original ozone, MDL, and qualifier text, together with the source file details and row fingerprint. This helps us interpret a reading and trace it back to Bronze.

`build_ozone_observations` uses the `finite_number` helper defined in **5.1**. It is called by `process_ozone_source` in **5.7** and returns a Spark DataFrame. The audit is built separately in **5.5**, so excluded records and extra copies remain accounted for.

In [ ]:
# Keep eligible ozone readings with quality flags and source details.
def build_ozone_observations(records):
    """Build the eligible ozone observation schema."""

    # Define how to rename the monitor identifier columns.
    identifier_columns = {
        "State Code": "state_code",
        "County Code": "county_code",
        "Site Num": "site_number",
        "Parameter Code": "parameter_code",
        "POC": "poc",
    }

    # Keep the detection limit only if it is finite and non-negative.
    # An unusable detection limit does not remove an otherwise eligible ozone reading.
    mdl_usable = (
        finite_number("mdl_value")
        & (F.col("mdl_value") >= 0)
    )

    # Describe retained readings with flags; these flags do not exclude the reading.
    flag_rules = {
        "negative_ozone": F.col("negative_ozone"),
        "below_reported_mdl": (
            mdl_usable & (F.col("ozone_value") < F.col("mdl_value"))
        ),
        "source_qualifier_XS": F.col("qualifier_code") == "XS",
        "source_qualifier_MD": F.col("qualifier_code") == "MD",
        "source_validation_qualifier": (
            F.col("qualifier_code").isin("1V", "V")
        ),
        "missing_mdl": F.col("missing_mdl"),
        "invalid_mdl": F.col("invalid_mdl"),
        "nonfinite_mdl": F.col("nonfinite_mdl"),
        "negative_mdl": F.col("negative_mdl"),
    }

    # Collect only the quality flags that apply to each row.
    quality_flags = F.filter(
        F.array(*[
            F.when(condition, F.lit(name))
            for name, condition in flag_rules.items()
        ]),
        lambda flag: flag.isNotNull(),
    )

    # Keep the columns that trace each row back to Bronze.
    lineage_columns = [
        "source_provider",
        "source_dataset",
        "source_version_sha256",
        "source_file_path",
        "source_file_sha256",
        "source_manifest_path",
        "source_record_hash_scheme",
        "source_record_sha256",
    ]

    # Return eligible readings with renamed columns, quality flags, and source details.
    return records.filter(F.col("record_eligible")).select(
        *[
            F.col(source_name).alias(output_name)
            for source_name, output_name in identifier_columns.items()
        ],
        F.concat_ws(
            "-", *[F.col(name) for name in identifier_columns]
        ).alias("monitor_id"),
        "timestamp_utc",
        F.col("ozone_value").alias("ozone_ppm"),
        F.col("Sample Measurement").alias("ozone_raw"),
        F.col("Units of Measure").alias("source_unit"),
        F.when(mdl_usable, F.col("mdl_value")).alias("mdl_ppm"),
        F.col("MDL").alias("mdl_raw"),
        "qualifier_code",
        F.col("Qualifier").alias("qualifier_raw"),
        F.col("Method Type").alias("method_type"),
        F.col("Method Code").alias("method_code"),
        F.col("Method Name").alias("method_name"),
        quality_flags.alias("quality_flags"),
        "source_occurrences",
        *lineage_columns,
    )


### 5.5 Account for excluded rows and extra copies

We build an audit explaining what happened to every selected source record. It includes accepted records, excluded records, and extra identical copies.

1. **Keep the evidence.** Preserve the original field values, quality-check results, failure reasons, and source details. The `named_column_map` helper pairs column names with their values. It is defined here in the notebook and called twice: once for the original fields and once for the check results.

2. **Record each distinct record's outcome.** Build the `representatives` table with one entry per distinct record, whether accepted or excluded. Each entry has a `record_count` of one.

3. **Account for extra copies.** Build a separate `duplicate_copies` table for records that appeared more than once. Each entry is labelled `exact_duplicate`, and its `record_count` gives the number of extra copies.

4. **Combine the entries.** Use `unionByName` to stack the two tables into one audit, matching columns by name.

One audit entry can represent several input copies, so we sum `record_count` to account for the input rather than just counting audit rows.

`build_ozone_audit` is called by `process_ozone_source` in **5.7**. That function checks that the audit counts match the selected input and that the accepted count matches the observation table.

In [ ]:
# Build the audit using maps of original values and quality-check results.
def named_column_map(column_names):
    """Build a Spark map pairing each requested column name with its value in the row."""
    entries = []
    for name in column_names:
        entries.extend([F.lit(name), F.col(name)])
    return F.create_map(*entries)


# Build the audit with outcomes and counts for accepted records, exclusions, and extra copies.
def build_ozone_audit(records, source_columns, diagnostic_columns):
    """Build counted audit entries for representatives and duplicate copies."""
    # List the columns used to build the monitor ID.
    monitor_columns = [
        "State Code", "County Code", "Site Num", "Parameter Code", "POC"
    ]

    # List the source details to retain in the audit
    lineage_columns = [
        "source_provider",
        "source_dataset",
        "source_version_sha256",
        "source_file_path",
        "source_file_sha256",
        "source_manifest_path",
        "source_record_hash_scheme",
        "source_record_sha256",
    ]

    # Start the audit with the selected columns for each distinct record
    base = (
        records
        .withColumn(
            "monitor_id",
            F.when(
                ~F.col("missing_identifier"),
                F.concat_ws("-", *[F.col(name) for name in monitor_columns]),
            ),
        )
        .withColumn("raw_fields", named_column_map(source_columns))
        .withColumn("field_flags", named_column_map(diagnostic_columns))
    )
    # List the audit fields shared by all outputs.
    common_columns = [
        "monitor_id",
        "timestamp_utc",
        "raw_fields",
        "field_flags",
        "field_disposition",
        "field_reasons",
        "key_variant_count",
        "key_conflict",
        *lineage_columns,
    ]

    # Record each distinct row's outcome and reasons, counting it once.
    representatives = base.select(
        *common_columns,
        F.col("record_disposition").alias("disposition"),
        F.col("record_reasons").alias("reasons"),
        F.lit(1).cast("long").alias("record_count"),
    )

    # Three identical inputs produce one representative and an extra-copy entry with count 2.
    # Summing record_count therefore still accounts for all three source records.
    duplicate_copies = (
        base
        .filter(F.col("duplicate_extra_records") > 0)
        .select(
            *common_columns,
            F.lit("exact_duplicate").alias("disposition"),
            F.concat(
                F.array(F.lit("exact_duplicate")),
                F.col("record_reasons"),
            ).alias("reasons"),
            F.col("duplicate_extra_records").alias("record_count"),
        )
    )

    # Combine the accepted records, exclusions, and extra copies into one audit table
    return representatives.unionByName(duplicate_copies)


### 5.6 Compare ozone before and after cleaning

This section prepares summaries that make the effects of cleaning visible.

1. **Calculate the summary metrics.** Count the records, negative ozone readings, and missing or unusable detection limits. Find the minimum and maximum finite ozone values and the earliest and latest timestamps. The result is one Python dictionary for the input table.

2. **Compare the input with the observations.** In **5.7**, `process_ozone_source` calls `summarize_ozone_values` twice: once for the selected input records and once for the eligible observation table. Using the same calculations makes the results comparable.

3. **Display the results.** In **5.8**, `show_source_summary` displays the existing before and after metrics, then prints the audit outcome counts and the quality-flag counts. It displays information; it does not return another data table.

Quality-flag counts can overlap. For example, one retained reading can be both negative and below the detection limit. Adding those two flag counts would count that reading twice. The audit outcome counts, by contrast, account for each selected source record once.

The summary calculations reuse `finite_number`, the notebook helper defined in **5.1**, to check numeric values.

In [ ]:
# Summarize counts, ozone ranges, detection limit issues, and the time span.
def summarize_ozone_values(records, ozone_column, mdl_column):
    """Return counts, value ranges and time bounds as a small Python dictionary."""
    ozone = F.col(ozone_column)
    ozone_is_finite = finite_number(ozone_column)
    mdl_is_usable = (
        finite_number(mdl_column)
        & (F.col(mdl_column) >= 0)
    )

    # Calculate one small summary row in Spark, then bring that row into Python.
    return records.agg(
        F.count("*").alias("records"),
        F.count(
            F.when(ozone_is_finite & (ozone < 0), 1)
        ).alias("negative_ozone"),
        F.count(
            F.when(~mdl_is_usable, 1)
        ).alias("missing_or_unusable_mdl"),
        F.min(
            F.when(ozone_is_finite, ozone)
        ).alias("minimum_ozone_ppm"),
        F.max(
            F.when(ozone_is_finite, ozone)
        ).alias("maximum_ozone_ppm"),
        F.date_format(
            F.min("timestamp_utc"), "yyyy-MM-dd HH:mm:ss"
        ).alias("earliest_timestamp_utc"),
        F.date_format(
            F.max("timestamp_utc"), "yyyy-MM-dd HH:mm:ss"
        ).alias("latest_timestamp_utc"),
    ).first().asDict()

# Display the small summaries already calculated by the processing function.
def show_source_summary(summary, metrics):
    """Display the supplied before/after summaries, audit totals and retained quality flags."""
    rows = [
        (
            label,
            str(summary["before"][key])
            if summary["before"][key] is not None else "N/A",
            str(summary["after"][key])
            if summary["after"][key] is not None else "N/A",
        )
        for label, key in metrics
    ]
    
    # Display the before and after metrics in a table.
    spark.createDataFrame(
        rows,
        schema="metric STRING, before_cleaning STRING, after_cleaning STRING",
    ).show(truncate=False)
    # Audit outcomes account for the input once; retained quality flags may overlap.
    print("Exclusive source-record dispositions:")
    for disposition, count in sorted(summary["dispositions"].items()):
        print(f"  {disposition}: {count}")
    print("Reconciled source records:", sum(summary["dispositions"].values()))
    print("Overlapping quality flags on retained observations:")
    if summary["quality_flags"]:
        for flag, count in sorted(summary["quality_flags"].items()):
            print(f"  {flag}: {count}")
    else:
        print("  None")


### 5.7 Process one monitor

This function connects the earlier notebook functions into one processing flow for the ozone monitor selected in the profile.

1. **Check the inputs.** Validate the profile's year and monitor identifiers. Confirm that the Bronze source and manifest describe EPA hourly ozone for that year, use compatible schemas, and stay inside the selected environment folder.

2. **Read and select the data.** Use the reader from **5.1**, check that the required columns exist, and select the records matching the monitor in the profile.

3. **Prepare and check the records.** Use **5.2** to convert values, add quality-check results, and classify each row. Then use **5.3** to attach source details, count identical copies, and flag conflicting records.

4. **Build the outputs and comparisons.** Pass the same resolved records to **5.4** for eligible observations and **5.5** for the complete audit. Use **5.6** to calculate summaries of the selected input and the resulting observations.

5. **Check the accounting.** Require the audit's total record count to match the selected input count. Also require its accepted-record count to match the observation table. Stop with an error if either comparison fails.

The function then counts the quality flags on retained observations and returns one dictionary containing the observation table, audit table, and summary. The summary includes the monitor and source identity, before and after metrics, audit outcome counts, and quality-flag counts.

This function is called for Alabama in **5.8**.

In [ ]:
# Process one ozone monitor and return observations, an audit, and a summary.
def process_ozone_source(source, profile, environment_root):
    """Build ozone observations, audit records and quality summaries."""
    year = profile["year"]
    monitor = profile["ozone_monitor"]
    monitor_fields = {
        "State Code": "state_code",
        "County Code": "county_code",
        "Site Num": "site_number",
        "Parameter Code": "parameter_code",
        "POC": "poc",
    }

    # Stop if the profile year is not an integer.
    if type(year) is not int:
        raise ValueError("The profile year must be an integer")

    # Read the monitor identifiers from the profile.
    # Require nonempty text without surrounding spaces.
    monitor_values = {
        column: monitor.get(profile_field)
        for column, profile_field in monitor_fields.items()
    }
    if any(
        not isinstance(value, str)
        or not value
        or value != value.strip()
        for value in monitor_values.values()
    ):
        raise ValueError("Monitor identifiers must be nonempty, trimmed strings")

    # Stop if the profile selects a parameter other than ozone.
    if monitor_values["Parameter Code"] != "44201":
        raise ValueError("The ozone entry point requires parameter code 44201")

    # Require an EPA hourly ozone source for the selected year.
    request = source["request"]
    if (
        request.get("source"),
        request.get("dataset"),
        request.get("year"),
    ) != ("epa", "hourly_ozone", year):
        raise ValueError("The Bronze source does not match the ozone profile")

    # Get full file paths and reject any outside the environment folder.
    root = Path(environment_root).resolve()
    data_path = Path(source["data_path"]).resolve()
    manifest_path = Path(source["manifest_path"]).resolve()

    for path in (data_path, manifest_path):
        if not path.is_relative_to(root):
            raise ValueError(f"Source path is outside the environment root: {path}")

    # Read the manifest and check its provider, dataset, and year.
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    expected_identity = {
        "provider": "epa",
        "dataset": "hourly_ozone",
        "year": year,
    }
    if any(
        manifest["source"].get(name) != value
        for name, value in expected_identity.items()
    ):
        raise ValueError("The Bronze manifest does not match the ozone profile")

    # Require both schema statuses to be compatible before reading the data.
    if (
        source["schema"]["status"] != "compatible"
        or manifest["schema"]["status"] != "compatible"
    ):
        raise ValueError("The Bronze ozone schema is not compatible")

    # Check the CSV structure and load its fields into Spark as text.
    raw, _ = read_validated_ozone_csv(data_path)

    # Define the required columns and stop if any are missing.
    required_columns = set(monitor_fields) | {
        "Date GMT",
        "Time GMT",
        "Sample Measurement",
        "Units of Measure",
        "MDL",
        "Qualifier",
        "Method Type",
        "Method Code",
        "Method Name",
    }
    missing_columns = required_columns - set(raw.columns)
    if missing_columns:
        raise ValueError(f"Missing ozone columns: {sorted(missing_columns)}")

    source_columns = raw.columns
    monitor_filter = F.lit(True)
    for column, value in monitor_values.items():
        monitor_filter = monitor_filter & (F.col(column) == F.lit(value))

    # Select only rows matching the monitor identifiers in the profile.
    selected = raw.filter(monitor_filter)

    # Add a timestamp and numeric ozone and MDL values; keep the original text.
    typed = type_ozone_fields(selected)

    # Add a true/false result for each quality check to every row.
    checked = typed.select(
        "*",
        *[
            condition.alias(name)
            for name, condition in ozone_field_checks.items()
        ],
    )

    # Mark eligibility, keep all failure reasons, and use the first as the summary.
    classified = classify_ozone_fields(checked, year=year)

    # Add source details and row fingerprints to trace results back to Bronze.
    with_lineage = attach_bronze_lineage(
        classified,
        source,
        root,
        source_columns,
    )

    # Count identical copies, keep one, and mark conflicting records as ineligible.
    resolved = resolve_source_keys(
        with_lineage,
        key_columns=[*monitor_fields, "timestamp_utc"],
        key_is_usable=(
            ~F.col("missing_identifier")
            & F.col("timestamp_utc").isNotNull()
        ),
    )

    # Build the eligible observation table with quality flags and source details.
    observations = build_ozone_observations(resolved)

    # Build the audit for accepted records, exclusions, and extra copies.
    audit = build_ozone_audit(
        resolved,
        source_columns,
        list(ozone_field_checks),
    )

    # Summarize the selected input before exclusions and duplicate removal.
    before = summarize_ozone_values(
        checked,
        ozone_column="ozone_value",
        mdl_column="mdl_value",
    )

    # Summarize eligible observations after exclusions and duplicate removal.
    after = summarize_ozone_values(
        observations,
        ozone_column="ozone_ppm",
        mdl_column="mdl_ppm",
    )

    # Total source records for each audit outcome, including extra copies.
    disposition_rows = (
        audit
        .groupBy("disposition")
        .agg(F.sum("record_count").alias("source_records"))
        .orderBy("disposition")
        .collect()
    )

    # Bring only the small outcome totals into Python for the accounting checks.
    dispositions = {
        row["disposition"]: row["source_records"]
        for row in disposition_rows
    }

    # Require all input records to be accounted for and the eligible total to match the output.
    if sum(dispositions.values()) != before["records"]:
        raise ValueError("Ozone audit counts do not reconcile with the input")

    if dispositions.get("eligible", 0) != after["records"]:
        raise ValueError("Ozone observation and eligible audit counts differ")

    # Count how many retained observations have each quality flag.
    flag_rows = (
        observations
        .select(F.explode("quality_flags").alias("quality_flag"))
        .groupBy("quality_flag")
        .agg(F.count("*").alias("observations"))
        .orderBy("quality_flag")
        .collect()
    )

    # A reading can have several flags, so these counts can overlap.
    quality_flags = {
        row["quality_flag"]: row["observations"]
        for row in flag_rows
    }

    # Return the observation table, audit table, and summary in one dictionary.
    return {
        "observations": observations,
        "audit": audit,
        "summary": {
            "monitor_id": "-".join(monitor_values.values()),
            "year": year,
            "source_version_sha256": manifest["original_file"]["sha256"],
            "before": before,
            "after": after,
            "dispositions": dispositions,
            "quality_flags": quality_flags,
        },
    }


### 5.8 Run the Alabama ozone check

Here are the results for McAdory in 2025.

In [ ]:
# Find the EPA hourly ozone source for the year in the Alabama profile.
ozone_candidates = [
    item for item in downloads
    if item["request"]["source"] == "epa"
    and item["request"]["dataset"] == "hourly_ozone"
    and item["request"]["year"] == alabama_profile["year"]
]

# Stop unless exactly one ozone source matches.
if len(ozone_candidates) != 1:
    raise ValueError("Expected one ozone input for the selected year.")

# Select the ozone source and stop if its schema is incompatible.
ozone_source = ozone_candidates[0]
if ozone_source["schema"]["status"] != "compatible":
    raise ValueError("The selected ozone input has a blocked schema.")

# Build observations, an audit, and a summary for the Alabama monitor.
ozone_result = process_ozone_source(
    source=ozone_source,
    profile=alabama_profile,
    environment_root=silver_context.environment_root,
)
# Keep the tables for publication and the small summary for display.
ozone_observations = ozone_result["observations"]
ozone_audit = ozone_result["audit"]
ozone_result_summary = ozone_result["summary"]

# Show the selected input version and the before/after cleaning results.
print("Ozone source:", ozone_source["data_path"])
print("Source version:", ozone_result_summary["source_version_sha256"])
ozone_summary_metrics = [
    ("Records", "records"),
    ("Negative ozone values", "negative_ozone"),
    ("Missing or unusable MDL", "missing_or_unusable_mdl"),
    ("Minimum finite ozone (ppm)", "minimum_ozone_ppm"),
    ("Maximum finite ozone (ppm)", "maximum_ozone_ppm"),
    ("Earliest timestamp (UTC)", "earliest_timestamp_utc"),
    ("Latest timestamp (UTC)", "latest_timestamp_utc"),
]
show_source_summary(ozone_result_summary, ozone_summary_metrics)

# Show a few retained readings only when extra diagnostics are enabled.
if show_diagnostics:
    ozone_observations.select(
        "monitor_id", "timestamp_utc", "ozone_ppm", "mdl_ppm", "quality_flags"
    ).orderBy("timestamp_utc").show(5, truncate=False)


26/11/13 23:13:36 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Ozone source: /home/ubuntu/dat535-final-project-data/dev/data/bronze/epa/hourly_ozone_2025/2b7b9e03b43a2bf20d3c3e78fc8c98b6b349fe2c3c4778382a2997144e7b2fd7/hourly_44201_2025.csv
Source version: 2b7b9e03b43a2bf20d3c3e78fc8c98b6b349fe2c3c4778382a2997144e7b2fd7


+--------------------------+-------------------+-------------------+
|metric                    |before_cleaning    |after_cleaning     |
+--------------------------+-------------------+-------------------+
|Records                   |5166               |5071               |
|Negative ozone values     |52                 |52                 |
|Missing or unusable MDL   |0                  |0                  |
|Minimum finite ozone (ppm)|-0.001             |-0.001             |
|Maximum finite ozone (ppm)|0.072              |0.072              |
|Earliest timestamp (UTC)  |2025-02-25 01:00:00|2025-04-01 06:00:00|
|Latest timestamp (UTC)    |2025-11-03 13:00:00|2025-11-03 13:00:00|
+--------------------------+-------------------+-------------------+

Exclusive source-record dispositions:
  eligible: 5071
  qualifier_6: 95
Reconciled source records: 5166
Overlapping quality flags on retained observations:
  below_reported_mdl: 922
  negative_ozone: 52


### 5.9 What changed for McAdory?

Of 5,166 source records, **5,071** passed the checks. All 95 exclusions had qualifier `6`. We found no duplicate or conflicting keys in this input.

The retained values ranged from −0.001 to 0.072 ppm. All 52 negative readings were among the 922 below the reported detection limit. Adding those counts would count the negative readings twice.

The first eligible hour was 1 April at 06:00 UTC and the last was 3 November at 13:00 UTC. Section 9 measures the gaps between and outside those dates.

## 6. Silver: temperature reports

### 6.1 Read temperatures and report times

NOAA supplies temperatures in degrees Celsius. We keep the actual UTC report time: a report at 14:53 stays at 14:53 until we select one report per hour in section 9.

This follows the same flow as ozone: convert fields, classify records, resolve repeated keys, then build observations and an audit. We select one report per clock hour later, after these source-level checks.

In [19]:
# List the NOAA fields that describe the temperature evidence for each report.
temperature_inspection_columns = [
    "STATION",
    "DATE",
    "temperature",
    "temperature_Measurement_Code",
    "temperature_Quality_Code",
    "temperature_Report_Type",
    "temperature_Source_Code",
    "temperature_Source_Station_ID",
]


def type_temperature_fields(records):
    """Add typed UTC timestamps and temperatures, retaining source fields."""
    if records.sparkSession.conf.get("spark.sql.session.timeZone") != "UTC":
        raise ValueError(
            "Temperature timestamp parsing requires a UTC Spark session."
        )

    return (
        records
        .withColumn(
            "timestamp_utc",
            F.try_to_timestamp(
                F.col("DATE"),
                F.lit("yyyy-MM-dd'T'HH:mm:ss"),
            ),
        )
        .withColumn(
            "temperature_value",
            F.expr("try_cast(`temperature` AS DOUBLE)"),
        )
    )


# Build checks that distinguish absent text from values that failed conversion.
temperature_missing_time = missing_string("DATE")
temperature_missing_value = missing_string("temperature")

temperature_field_checks = {
    "missing_station": missing_string("STATION"),
    "missing_timestamp": temperature_missing_time,
    "invalid_timestamp": (
        ~temperature_missing_time
        & F.col("timestamp_utc").isNull()
    ),
    "missing_temperature": temperature_missing_value,
    "invalid_temperature": (
        ~temperature_missing_value
        & F.col("temperature_value").isNull()
    ),
    "nonfinite_temperature": (
        F.col("temperature_value").isNotNull()
        & ~finite_number("temperature_value")
    ),
}


### 6.2 Apply the temperature quality rules

We accept finite temperatures from −100°C to 60°C inclusive. We also check the station, year and NOAA codes. The supported source/quality pairs are `223/1` and `343/5`, with a blank measurement code and report type `FM12`, `FM15` or `FM16`.

In [20]:
# Classify temperature reports while retaining every source row for the audit.
def classify_temperature_fields(
    records,
    year,
    station_id,
    source_quality_pairs,
):
    """Add eligibility and failure reasons using the selected station, year and NOAA policy."""
    # Read the profile policy and restrict it to source/quality pairs we have reviewed.
    accepted_pairs = {
        (str(source).strip(), str(quality).strip())
        for source, quality in source_quality_pairs
    }
    reviewed_pairs = {("223", "1"), ("343", "5")}

    if not accepted_pairs or not accepted_pairs <= reviewed_pairs:
        raise ValueError(
            "Temperature source/quality pairs must be a nonempty subset "
            "of the reviewed pairs: 223/1 and 343/5."
        )

    # Normalize metadata for comparison while retaining the source fields.
    source = F.coalesce(
        F.trim(F.col("temperature_Source_Code")), F.lit("")
    )
    quality = F.coalesce(
        F.trim(F.col("temperature_Quality_Code")), F.lit("")
    )
    report = F.coalesce(
        F.trim(F.col("temperature_Report_Type")), F.lit("")
    )
    measurement = F.coalesce(
        F.trim(F.col("temperature_Measurement_Code")), F.lit("")
    )

    timestamp = F.col("timestamp_utc")
    value = F.col("temperature_value")
    finite = finite_number("temperature_value")

    # Match the source and quality code together, because quality codes depend on the source.
    matches_pair = F.lit(False)
    for pair_source, pair_quality in sorted(accepted_pairs):
        matches_pair = matches_pair | (
            (source == pair_source) & (quality == pair_quality)
        )

    # The same quality code can mean different things for different NOAA sources.
    rejected_quality = (
        ((source == "343") & quality.isin("2", "3", "6", "7"))
        | ((source == "223") & quality.isin("2", "3", "5"))
    )

    # Check station, time, measurement and metadata. The first failure is the summary reason.
    rules = [
        ("missing_station", F.col("missing_station")),
        (
            "unexpected_station",
            ~F.col("missing_station")
            & (F.col("STATION") != F.lit(station_id)),
        ),
        ("missing_timestamp", F.col("missing_timestamp")),
        ("invalid_timestamp", F.col("invalid_timestamp")),
        (
            "outside_year",
            timestamp.isNotNull() & (F.year(timestamp) != year),
        ),
        ("missing_temperature", F.col("missing_temperature")),
        ("invalid_temperature", F.col("invalid_temperature")),
        ("nonfinite_temperature", F.col("nonfinite_temperature")),
        (
            "outside_temperature_range",
            finite & ~value.between(-100.0, 60.0),
        ),
        ("rejected_source_quality", rejected_quality),
        (
            "unsupported_source_quality",
            ~matches_pair & ~rejected_quality,
        ),
        (
            "unsupported_report_type",
            ~report.isin("FM12", "FM15", "FM16"),
        ),
        ("unsupported_measurement_code", measurement != ""),
    ]

    # Keep all applicable failure reasons, leaving out checks that passed.
    reasons = F.filter(
        F.array(*[
            F.when(condition, F.lit(reason))
            for reason, condition in rules
        ]),
        lambda reason: reason.isNotNull(),
    )

    # Return normalised codes, all reasons and the eligibility decision without dropping rows.
    return (
        records
        .withColumn("source_code", source)
        .withColumn("quality_code", quality)
        .withColumn("report_type", report)
        .withColumn("measurement_code", measurement)
        .withColumn("field_reasons", reasons)
        .withColumn("field_eligible", F.size("field_reasons") == 0)
        .withColumn(
            "field_disposition",
            F.when(F.col("field_eligible"), F.lit("eligible"))
            .otherwise(F.element_at("field_reasons", 1)),
        )
    )


### 6.3 Resolve reports at the same timestamp

For a given station and time, exact copies match the whole row. We keep one copy and record how many there were. Reports can also agree on the temperature fields while differing in other weather fields. We keep one representative of that temperature evidence and account for the others in the audit.

The comparison includes the temperature value and its source, quality and report codes. Equal temperatures alone do not make the reports equivalent. Conflicting temperature evidence is excluded. The representative is chosen using a stable ordering so repeated runs make the same choice.

In [21]:
# Resolve repeated station/timestamp keys using the temperature evidence.
def resolve_temperature_keys(records, source_columns, temperature_columns):
    """Count copies, reject conflicting temperature evidence and flag redundant reports."""
    if (
        not temperature_columns
        or not set(temperature_columns) <= set(source_columns)
    ):
        raise ValueError(
            "Temperature comparison fields must be original source columns."
        )

    # Only compare reports from the same source file and version.
    source_scope = [
        "source_provider",
        "source_dataset",
        "source_version_sha256",
        "source_file_path",
        "source_file_sha256",
    ]
    key_columns = ["STATION", "timestamp_utc"]

    # A missing station or timestamp cannot form a usable report key.
    usable_key = F.coalesce(
        ~F.col("missing_station")
        & F.col("timestamp_utc").isNotNull(),
        F.lit(False),
    )

    # Collapse complete copies, including their stable derived fields.
    distinct_records = records.groupBy(*records.columns).agg(
        F.count("*").alias("source_occurrences")
    )

    key_window = Window.partitionBy(*(source_scope + key_columns))
    # Compare the temperature value AND its source, quality and report metadata.
    # Equal temperatures alone do not mean two reports provide the same evidence.
    temperature_window = key_window.orderBy(
        *[F.col(name) for name in temperature_columns]
    )

    # Use the full original row as a stable tie-breaker when choosing a representative.
    source_json = F.to_json(
        F.struct(*[F.col(name) for name in sorted(source_columns)]),
        options={"ignoreNullFields": "false"},
    )
    representative_window = key_window.orderBy(
        F.col("source_record_sha256"),
        source_json,
    )

    # dense_rank gives equal temperature evidence the same rank.
    # row_number chooses one full source row when the evidence agrees.
    ranked = (
        distinct_records
        .withColumn(
            "_temperature_rank",
            F.dense_rank().over(temperature_window),
        )
        .withColumn(
            "_source_rank",
            F.row_number().over(representative_window),
        )
    )

    # The highest dense rank tells us how many different temperature variants exist.
    # Exclude conflicting variants. When eligible reports agree, keep the first
    # representative and label the others redundant_temperature for the audit.
    resolved = (
        ranked
        .withColumn(
            "duplicate_extra_records",
            F.col("source_occurrences") - 1,
        )
        .withColumn(
            "key_variant_count",
            F.when(
                usable_key,
                F.max("_temperature_rank").over(key_window),
            ),
        )
        .withColumn(
            "key_source_records",
            F.when(
                usable_key,
                F.sum("source_occurrences").over(key_window),
            ),
        )
        .withColumn(
            "key_conflict",
            F.coalesce(
                F.col("key_variant_count") > 1,
                F.lit(False),
            ),
        )
        .withColumn(
            "redundant_temperature",
            usable_key
            & ~F.col("key_conflict")
            & F.col("field_eligible")
            & (F.col("_source_rank") > 1),
        )
        .withColumn(
            "record_disposition",
            F.when(
                F.col("key_conflict"),
                F.lit("conflicting_key"),
            )
            .when(
                F.col("redundant_temperature"),
                F.lit("redundant_temperature"),
            )
            .otherwise(F.col("field_disposition")),
        )
        .withColumn(
            "record_reasons",
            F.when(
                F.col("key_conflict"),
                F.concat(
                    F.array(F.lit("conflicting_key")),
                    F.col("field_reasons"),
                ),
            )
            .when(
                F.col("redundant_temperature"),
                F.concat(
                    F.array(F.lit("redundant_temperature")),
                    F.col("field_reasons"),
                ),
            )
            .otherwise(F.col("field_reasons")),
        )
        .withColumn(
            "record_eligible",
            F.col("record_disposition") == "eligible",
        )
        .drop("_temperature_rank", "_source_rank")
    )

    # Return the decisions and copy counts for both the observation table and audit.
    return resolved


### 6.4 Build the temperature and audit tables

Each eligible report keeps its temperature, timestamp and source details. The audit also accounts for rejected reports and duplicate copies.

In [22]:
# Build the eligible report table and the separate audit from the same resolved rows.
def build_temperature_observations(records):
    """Return eligible temperature reports with their original times, flags and source details."""
    # A missing original station code is a retained quality flag, not an exclusion.
    missing_original_station = missing_string(
        "temperature_Source_Station_ID"
    )

    quality_flags = F.filter(
        F.array(
            F.when(
                missing_original_station,
                F.lit("missing_source_station_id"),
            ),
        ),
        lambda flag: flag.isNotNull(),
    )

    lineage_columns = [
        "source_provider",
        "source_dataset",
        "source_version_sha256",
        "source_file_path",
        "source_file_sha256",
        "source_manifest_path",
        "source_record_hash_scheme",
        "source_record_sha256",
    ]

    # Keep the actual report time, Celsius value, original fields and Bronze source details.
    return records.filter(F.col("record_eligible")).select(
        F.col("STATION").alias("station_id"),
        "timestamp_utc",
        F.col("temperature_value").alias("temperature_c"),
        F.col("temperature").alias("temperature_raw"),
        F.col("DATE").alias("timestamp_raw"),
        "source_code",
        "quality_code",
        "report_type",
        "measurement_code",
        F.when(
            ~missing_original_station,
            F.trim(F.col("temperature_Source_Station_ID")),
        ).alias("source_station_id"),
        F.col("temperature_Source_Code").alias("source_code_raw"),
        F.col("temperature_Quality_Code").alias("quality_code_raw"),
        F.col("temperature_Report_Type").alias("report_type_raw"),
        F.col("temperature_Measurement_Code").alias("measurement_code_raw"),
        F.col("temperature_Source_Station_ID").alias("source_station_id_raw"),
        quality_flags.alias("quality_flags"),
        "source_occurrences",
        F.col("key_source_records").alias("supporting_source_records"),
        *lineage_columns,
    )


def build_temperature_audit(records, source_columns, diagnostic_columns):
    """Build counted audit entries for source rows and duplicate copies."""
    lineage_columns = [
        "source_provider",
        "source_dataset",
        "source_version_sha256",
        "source_file_path",
        "source_file_sha256",
        "source_manifest_path",
        "source_record_hash_scheme",
        "source_record_sha256",
    ]

    # Attach raw field values and check results so each audit entry can be explained.
    base = (
        records
        .withColumn(
            "station_id",
            F.when(
                ~F.col("missing_station"),
                F.col("STATION"),
            ),
        )
        .withColumn(
            "raw_fields",
            named_column_map(source_columns),
        )
        .withColumn(
            "field_flags",
            named_column_map(diagnostic_columns),
        )
    )

    common_columns = [
        "station_id",
        "timestamp_utc",
        "raw_fields",
        "field_flags",
        "field_disposition",
        "field_reasons",
        "key_variant_count",
        "key_source_records",
        "key_conflict",
        "redundant_temperature",
        *lineage_columns,
    ]

    # Count each distinct source row once, whether accepted or excluded.
    representatives = base.select(
        *common_columns,
        F.col("record_disposition").alias("disposition"),
        F.col("record_reasons").alias("reasons"),
        F.lit(1).cast("long").alias("record_count"),
    )

    # Account for extra identical copies separately using their copy counts.
    duplicate_copies = (
        base
        .filter(F.col("duplicate_extra_records") > 0)
        .select(
            *common_columns,
            F.lit("exact_duplicate").alias("disposition"),
            F.concat(
                F.array(F.lit("exact_duplicate")),
                F.col("record_reasons"),
            ).alias("reasons"),
            F.col("duplicate_extra_records").alias("record_count"),
        )
    )

    # Combine distinct-record outcomes and extra copies into one complete audit.
    return representatives.unionByName(duplicate_copies)


### 6.5 Process one station

`process_temperature_source` applies these rules to one station-year file. It checks that the audit accounts for the input and agrees with the number of eligible reports.

The function returns the observation DataFrame, audit DataFrame and a small summary. The cell in 6.6 calls it for Alabama; defining the function here does not process the file yet.

In [23]:
# Define the summary and the main station-processing function used below.
def summarize_temperature_values(
    records,
    temperature_column,
    source_station_column,
):
    """Summarize temperature values, metadata completeness and time bounds."""
    temperature = F.col(temperature_column)
    temperature_is_finite = finite_number(temperature_column)

    # Calculate comparable before/after counts, ranges and the observed time span.
    return records.agg(
        F.count("*").alias("records"),
        F.count(
            F.when(~temperature_is_finite, 1)
        ).alias("missing_or_unusable_temperature"),
        F.count(
            F.when(
                temperature_is_finite & (temperature < 0),
                1,
            )
        ).alias("negative_temperatures"),
        F.count(
            F.when(missing_string(source_station_column), 1)
        ).alias("missing_source_station_id"),
        F.min(
            F.when(temperature_is_finite, temperature)
        ).alias("minimum_temperature_c"),
        F.max(
            F.when(temperature_is_finite, temperature)
        ).alias("maximum_temperature_c"),
        F.date_format(
            F.min("timestamp_utc"),
            "yyyy-MM-dd HH:mm:ss",
        ).alias("earliest_timestamp_utc"),
        F.date_format(
            F.max("timestamp_utc"),
            "yyyy-MM-dd HH:mm:ss",
        ).alias("latest_timestamp_utc"),
    ).first().asDict()


def process_temperature_source(source, profile, environment_root):
    """Build temperature observations, audit records and quality summaries."""
    # Read the selected year, station and temperature policy from the profile.
    year = profile["year"]
    station_id = profile["weather_mapping"]["noaa_station_id"]
    source_quality_pairs = (
        profile["temperature_policy"]["source_quality_pairs"]
    )

    # Reject incomplete profile identifiers before opening a source file.
    if type(year) is not int:
        raise ValueError("The profile year must be an integer.")

    if (
        not isinstance(station_id, str)
        or not station_id
        or station_id != station_id.strip()
    ):
        raise ValueError(
            "The station identifier must be a nonempty, trimmed string."
        )

    # Check that the selected Bronze input belongs to this station and year.
    request = source["request"]
    if (
        request.get("source"),
        request.get("dataset"),
        request.get("year"),
        request.get("station_id"),
    ) != ("noaa", "hourly_weather", year, station_id):
        raise ValueError(
            "The Bronze source does not match the temperature profile."
        )

    # Resolve paths and require the inputs to stay inside this environment.
    root = Path(environment_root).resolve(strict=True)
    data_path = Path(source["data_path"]).resolve(strict=True)
    manifest_path = Path(source["manifest_path"]).resolve(strict=True)

    for path in (data_path, manifest_path):
        if not path.is_relative_to(root):
            raise ValueError(
                f"Source path is outside the environment root: {path}"
            )

    # Confirm the saved manifest identifies the same source and has a compatible schema.
    manifest = json.loads(
        manifest_path.read_text(encoding="utf-8")
    )

    if manifest.get("manifest_version") != 1:
        raise ValueError("Unsupported Bronze manifest version.")

    expected_identity = {
        "provider": "noaa",
        "dataset": "hourly_weather",
        "year": year,
        "station_id": station_id,
    }
    if any(
        manifest["source"].get(name) != value
        for name, value in expected_identity.items()
    ):
        raise ValueError(
            "The Bronze manifest does not match the temperature profile."
        )

    if (
        source["schema"]["status"] != "compatible"
        or manifest["schema"]["status"] != "compatible"
    ):
        raise ValueError(
            "The Bronze temperature schema is not compatible."
        )

    # Read the pipe-delimited file as text and keep the independent structural row count.
    raw, structure = read_validated_delimited_file(
        data_path,
        delimiter="|",
    )

    # Require all fields used to decide whether temperature evidence agrees.
    temperature_columns = [
        "STATION",
        "DATE",
        "temperature",
        "temperature_Measurement_Code",
        "temperature_Quality_Code",
        "temperature_Report_Type",
        "temperature_Source_Code",
        "temperature_Source_Station_ID",
    ]

    missing_columns = set(temperature_columns) - set(raw.columns)
    if missing_columns:
        raise ValueError(
            f"Missing temperature columns: {sorted(missing_columns)}"
        )

    source_columns = raw.columns

    # Keep every source row while adding typed values and check results; build the audit later.
    typed = type_temperature_fields(raw)
    checked = typed.select(
        "*",
        *[
            condition.alias(name)
            for name, condition in temperature_field_checks.items()
        ],
    )

    # Apply the station, year, value and NOAA metadata rules.
    classified = classify_temperature_fields(
        checked,
        year=year,
        station_id=station_id,
        source_quality_pairs=source_quality_pairs,
    )

    # Attach the source-file identity and original-row fingerprint.
    with_lineage = attach_bronze_lineage(
        classified,
        source=source,
        environment_root=root,
        source_columns=source_columns,
    )

    # Resolve exact copies, conflicting evidence and redundant reports at the same time.
    resolved = resolve_temperature_keys(
        with_lineage,
        source_columns=source_columns,
        temperature_columns=temperature_columns,
    )

    # Build both outputs from the same decisions so accepted and excluded records reconcile.
    observations = build_temperature_observations(resolved)
    audit = build_temperature_audit(
        resolved,
        source_columns=source_columns,
        diagnostic_columns=list(temperature_field_checks),
    )

    # Compare the full input with the retained observations using the same metrics.
    before = summarize_temperature_values(
        checked,
        temperature_column="temperature_value",
        source_station_column="temperature_Source_Station_ID",
    )
    after = summarize_temperature_values(
        observations,
        temperature_column="temperature_c",
        source_station_column="source_station_id",
    )

    # Sum record_count so an audit entry representing several copies contributes all of them.
    disposition_rows = (
        audit
        .groupBy("disposition")
        .agg(F.sum("record_count").alias("source_records"))
        .orderBy("disposition")
        .collect()
    )
    dispositions = {
        row["disposition"]: row["source_records"]
        for row in disposition_rows
    }

    # Check the reader count, complete audit accounting and eligible output count.
    if before["records"] != structure["records"]:
        raise ValueError(
            "Spark and structural temperature record counts differ."
        )

    if sum(dispositions.values()) != before["records"]:
        raise ValueError(
            "Temperature audit counts do not reconcile with the input."
        )

    if dispositions.get("eligible", 0) != after["records"]:
        raise ValueError(
            "Temperature observation and eligible audit counts differ."
        )

    if after["missing_or_unusable_temperature"] != 0:
        raise ValueError(
            "Eligible observations contain unusable temperatures."
        )

    # Count retained quality flags and compare the missing-source-station count with the summary.
    flag_rows = (
        observations
        .select(F.explode("quality_flags").alias("quality_flag"))
        .groupBy("quality_flag")
        .agg(F.count("*").alias("observations"))
        .orderBy("quality_flag")
        .collect()
    )
    quality_flags = {
        row["quality_flag"]: row["observations"]
        for row in flag_rows
    }

    if (
        quality_flags.get("missing_source_station_id", 0)
        != after["missing_source_station_id"]
    ):
        raise ValueError(
            "Missing source-station flags and summary counts differ."
        )

    # Return the two Spark tables and a small summary for display and validation.
    return {
        "observations": observations,
        "audit": audit,
        "summary": {
            "station_id": station_id,
            "year": year,
            "source_version_sha256": manifest["original_file"]["sha256"],
            "before": before,
            "after": after,
            "dispositions": dispositions,
            "quality_flags": quality_flags,
        },
    }


### 6.6 Run the Alabama temperature check

We use the 2025 weather file for the Alabaster station.

In [24]:
# Select the weather file for the Alabama station and year.
temperature_station_id = (
    alabama_profile["weather_mapping"]["noaa_station_id"]
)

temperature_candidates = [
    item for item in downloads
    if item["request"]["source"] == "noaa"
    and item["request"]["dataset"] == "hourly_weather"
    and item["request"].get("year") == alabama_profile["year"]
    and item["request"].get("station_id") == temperature_station_id
]

# Stop rather than choosing between missing or ambiguous source selections.
if len(temperature_candidates) != 1:
    raise ValueError(
        "Expected one weather input for the selected station and year."
    )

temperature_source = temperature_candidates[0]
if temperature_source["schema"]["status"] != "compatible":
    raise ValueError("The selected weather input has a blocked schema.")

# Run the shared station processing and keep its outputs for publication.
temperature_result = process_temperature_source(
    source=temperature_source,
    profile=alabama_profile,
    environment_root=silver_context.environment_root,
)
temperature_observations = temperature_result["observations"]
temperature_audit = temperature_result["audit"]
temperature_result_summary = temperature_result["summary"]

# Show the source version and the before/after temperature summary.
print("Temperature source:", temperature_source["data_path"])
print("Source version:", temperature_result_summary["source_version_sha256"])
temperature_summary_metrics = [
    ("Records", "records"),
    ("Missing or unusable temperature", "missing_or_unusable_temperature"),
    ("Temperatures below zero (C)", "negative_temperatures"),
    ("Missing original source-station ID", "missing_source_station_id"),
    ("Minimum finite temperature (C)", "minimum_temperature_c"),
    ("Maximum finite temperature (C)", "maximum_temperature_c"),
    ("Earliest timestamp (UTC)", "earliest_timestamp_utc"),
    ("Latest timestamp (UTC)", "latest_timestamp_utc"),
]
show_source_summary(temperature_result_summary, temperature_summary_metrics)

# Optionally display sample reports; their timestamps have not been rounded to hours.
if show_diagnostics:
    temperature_observations.select(
        "station_id", "timestamp_utc", "temperature_c",
        "source_code", "quality_code", "report_type", "quality_flags",
    ).orderBy("timestamp_utc").show(5, truncate=False)


26/11/13 23:14:42 WARN DAGScheduler: Broadcasting large task binary with size 1041.5 KiB
26/11/13 23:14:43 WARN DAGScheduler: Broadcasting large task binary with size 1094.9 KiB
26/11/13 23:14:44 WARN DAGScheduler: Broadcasting large task binary with size 1095.5 KiB
26/11/13 23:14:44 WARN DAGScheduler: Broadcasting large task binary with size 1091.6 KiB


Temperature source: /home/ubuntu/dat535-final-project-data/dev/data/bronze/noaa/hourly_weather_2025_USW00053864/2ccd886d5e6ba9b8c502058db047094dc8a76e50bcfd1c3d54f5c458ab97d308/GHCNh_USW00053864_2025.psv
Source version: 2ccd886d5e6ba9b8c502058db047094dc8a76e50bcfd1c3d54f5c458ab97d308
+----------------------------------+-------------------+-------------------+
|metric                            |before_cleaning    |after_cleaning     |
+----------------------------------+-------------------+-------------------+
|Records                           |11851              |11686              |
|Missing or unusable temperature   |29                 |0                  |
|Temperatures below zero (C)       |365                |365                |
|Missing original source-station ID|29                 |0                  |
|Minimum finite temperature (C)    |-10.0              |-10.0              |
|Maximum finite temperature (C)    |36.7               |36.7               |
|Earliest timestamp (U

### 6.7 What changed for Alabaster?

Of 11,851 weather reports, **11,686** passed. There were 29 missing temperatures, 7 rejected source/quality combinations and 129 unsupported combinations. We found no repeated station–timestamp keys in this file.

The retained temperatures ranged from −10.0°C to 36.7°C, including 365 readings below zero. Several reports can fall in the same hour, so 11,686 reports represent fewer than 11,686 hours.

## 7. Silver: site, monitor and station records

### 7.1 Select the three reference records

We use the profile identifiers to find the Alabama site, ozone monitor and weather station. The Georgia check later uses the same original reference files with its own identifiers.

These are metadata about the locations and instruments, rather than hourly measurements. We use the same processing steps for all three reference types and keep an audit of their selected source rows.

In [25]:
# Use the Alabama profile to select its site, ozone monitor and weather station.
reference_site_keys = {
    "State Code": alabama_profile["ozone_monitor"]["state_code"],
    "County Code": alabama_profile["ozone_monitor"]["county_code"],
    "Site Number": alabama_profile["ozone_monitor"]["site_number"],
}

# Describe which provider, dataset and identifiers select each reference type.
reference_specs = [
    (
        "epa_sites",
        "epa",
        "sites",
        reference_site_keys,
    ),
    (
        "epa_monitors",
        "epa",
        "monitors",
        {
            **reference_site_keys,
            "Parameter Code": (
                alabama_profile["ozone_monitor"]["parameter_code"]
            ),
            "POC": alabama_profile["ozone_monitor"]["poc"],
        },
    ),
    (
        "noaa_stations",
        "noaa",
        "stations",
        {
            "GHCN_ID": (
                alabama_profile["weather_mapping"]["noaa_station_id"]
            ),
        },
    ),
]

# Keep the source details, full raw tables and selected profile rows separately.
reference_sources = {}
reference_raw_tables = {}
reference_profile_rows = {}

for (
    reference_name,
    reference_provider,
    reference_dataset,
    reference_keys,
) in reference_specs:
    # Find exactly one selected Bronze input for this reference type.
    reference_candidates = [
        item for item in downloads
        if item["request"]["source"] == reference_provider
        and item["request"]["dataset"] == reference_dataset
    ]

    if len(reference_candidates) != 1:
        raise ValueError(
            f"{reference_name}: expected one selected Bronze input."
        )

    reference_source = reference_candidates[0]

    if reference_source["schema"]["status"] != "compatible":
        raise ValueError(
            f"{reference_name}: the selected input has a blocked schema."
        )

    # Read the source with the same structural checks used for measurement files.
    reference_records, reference_structure = (
        read_validated_delimited_file(
            reference_source["data_path"],
            delimiter=",",
        )
    )

    # Require the reference columns needed by the pipeline.
    reference_missing_columns = (
        set(required_columns[(reference_provider, reference_dataset)])
        - set(reference_records.columns)
    )
    if reference_missing_columns:
        raise ValueError(
            f"{reference_name}: missing required columns "
            f"{sorted(reference_missing_columns)}"
        )

    # Compare Spark's row count with the structural reader before selecting profile rows.
    reference_record_count = reference_records.count()

    if reference_record_count != reference_structure["records"]:
        raise ValueError(
            f"{reference_name}: structural and Spark record counts differ."
        )

    # Combine the profile identifiers into one filter for this site, monitor or station.
    reference_match = F.lit(True)
    for source_column, expected_value in reference_keys.items():
        reference_match = reference_match & (
            F.col(source_column) == F.lit(expected_value)
        )

    reference_sources[reference_name] = reference_source
    # Georgia later filters the original reference tables using its own keys.
    reference_raw_tables[reference_name] = reference_records
    reference_profile_rows[reference_name] = reference_records.filter(reference_match)

### 7.2 Parse coordinates, elevations and dates

We convert the fields needed to check the pairing and keep the original identifiers. For example, a latitude outside −90° to 90° cannot describe a usable location.

In [26]:
# Map each original reference field to the output name and type we need.
reference_type_specs = {
    "epa_sites": {
        "latitude_deg": ("Latitude", "double"),
        "longitude_deg": ("Longitude", "double"),
        "elevation_m": ("Elevation", "double"),
        "site_established_date": ("Site Established Date", "date"),
        "site_closed_date": ("Site Closed Date", "date"),
        "extraction_date": ("Extraction Date", "date"),
    },
    "epa_monitors": {
        "latitude_deg": ("Latitude", "double"),
        "longitude_deg": ("Longitude", "double"),
        "first_year_of_data": ("First Year of Data", "int"),
        "last_sample_date": ("Last Sample Date", "date"),
        "extraction_date": ("Extraction Date", "date"),
    },
    "noaa_stations": {
        "latitude_deg": ("LATITUDE", "double"),
        "longitude_deg": ("LONGITUDE", "double"),
        "elevation_m": ("ELEVATION", "double"),
    },
}


def type_reference_fields(records, reference_name):
    """Add typed reference fields while preserving original source columns."""
    if reference_name not in reference_type_specs:
        raise ValueError(
            f"Unsupported reference type: {reference_name}"
        )

    # Use the field list for the selected reference type and require those source columns.
    field_specs = reference_type_specs[reference_name]

    missing_columns = {
        source_name
        for source_name, _ in field_specs.values()
    } - set(records.columns)

    if missing_columns:
        raise ValueError(
            f"{reference_name}: missing columns "
            f"{sorted(missing_columns)}"
        )

    # Avoid overwriting source columns when adding typed fields.
    existing_names = {name.casefold() for name in records.columns}
    collisions = [
        name for name in field_specs
        if name.casefold() in existing_names
    ]
    if collisions:
        raise ValueError(
            f"Typed fields would overwrite existing columns: {collisions}"
        )

    # Build date and number conversions; failed conversions become null.
    typed_expressions = []

    for output_name, (source_name, target_type) in field_specs.items():
        if target_type == "date":
            parsed_value = F.try_to_timestamp(
                F.col(source_name),
                F.lit("yyyy-MM-dd"),
            ).cast("date")
        elif target_type in {"double", "int"}:
            escaped_source_name = source_name.replace("`", "``")
            parsed_value = F.expr(
                f"try_cast(`{escaped_source_name}` AS {target_type})"
            )
        else:
            raise ValueError(
                f"Unsupported reference field type: {target_type}"
            )

        typed_expressions.append(
            parsed_value.alias(output_name)
        )

    # Keep the original text alongside the converted values for the audit.
    return records.select("*", *typed_expressions)


def build_reference_field_checks(reference_name, key_columns):
    """Build reference diagnostics without scanning source records."""
    # Start with missing identifiers, then add checks for the typed reference fields.
    checks = {
        f"missing_identifier_{name.lower().replace(' ', '_')}": missing_string(name)
        for name in key_columns
    }

    # Compare original values with their typed counterparts.
    for typed_column, (source_column, data_type) in (
        reference_type_specs[reference_name].items()
    ):
        source_missing = missing_string(source_column)

        checks[f"missing_{typed_column}"] = source_missing
        checks[f"invalid_{typed_column}"] = (
            ~source_missing & F.col(typed_column).isNull()
        )

        if data_type == "double":
            checks[f"nonfinite_{typed_column}"] = (
                F.col(typed_column).isNotNull()
                & ~finite_number(typed_column)
            )

    # Latitude and longitude must describe a possible location on Earth.
    checks["latitude_out_of_range"] = (
        finite_number("latitude_deg")
        & ~F.col("latitude_deg").between(-90.0, 90.0)
    )
    checks["longitude_out_of_range"] = (
        finite_number("longitude_deg")
        & ~F.col("longitude_deg").between(-180.0, 180.0)
    )

    if reference_name in {"epa_sites", "epa_monitors"}:
        checks["missing_datum"] = missing_string("Datum")

    # Compare metadata dates without restricting them to the analysis year.
    if reference_name == "epa_sites":
        checks["closure_before_establishment"] = (
            F.col("site_closed_date") < F.col("site_established_date")
        )
        checks["establishment_after_extraction"] = (
            F.col("site_established_date") > F.col("extraction_date")
        )
        checks["closure_after_extraction"] = (
            F.col("site_closed_date") > F.col("extraction_date")
        )

    elif reference_name == "epa_monitors":
        checks["first_year_after_last_sample"] = (
            F.col("first_year_of_data") > F.year("last_sample_date")
        )
        checks["last_sample_after_extraction"] = (
            F.col("last_sample_date") > F.col("extraction_date")
        )
    return checks


# Prepare the Spark check expressions for each reference type; rows are checked later.
reference_field_checks = {
    reference_name: build_reference_field_checks(reference_name, reference_keys)
    for reference_name, _, _, reference_keys in reference_specs
}


### 7.3 Decide which reference records are usable

A missing identifier or unusable coordinate excludes the record. Problems with optional fields are flagged. NOAA uses `-999.9` for missing elevation. An EPA coordinate datum other than WGS84 needs review before we interpret the distance between stations.

In [27]:
# Separate problems that exclude a reference from optional metadata warnings.
def classify_reference_fields(records, reference_name):
    """Add exclusion reasons and optional quality flags to reference records."""
    if reference_name not in reference_field_checks:
        raise ValueError(f"Unknown reference: {reference_name}")

    # Copy the shared checks so reference-specific additions do not change the original set.
    checks = dict(reference_field_checks[reference_name])

    # NOAA documents -999.9 as missing elevation.
    if reference_name == "noaa_stations":
        checks["missing_elevation_m"] = (
            missing_string("ELEVATION")
            | F.coalesce(
                F.col("elevation_m") == F.lit(-999.9),
                F.lit(False),
            )
        )

    # Keep the stated coordinate reference system (datum). Flag non-WGS84 coordinates
    # for review before using them to judge the distance between the stations.
    if reference_name in {"epa_sites", "epa_monitors"}:
        checks["non_wgs84_datum"] = (
            ~missing_string("Datum")
            & (F.upper(F.trim(F.col("Datum"))) != F.lit("WGS84"))
        )

    # Flag impossible first-year values and dates that disagree with the extraction date.
    if reference_name == "epa_monitors":
        checks["first_year_out_of_range"] = (
            F.col("first_year_of_data").isNotNull()
            & ~F.col("first_year_of_data").between(1, 9999)
        )
        checks["first_year_after_extraction"] = (
            F.col("first_year_of_data") > F.year("extraction_date")
        )

    # Missing identifiers or unusable coordinates prevent this reference from being used.
    exclusion_names = [
        name
        for name in checks
        if name.startswith("missing_identifier_")
    ] + [
        "missing_latitude_deg",
        "invalid_latitude_deg",
        "nonfinite_latitude_deg",
        "latitude_out_of_range",
        "missing_longitude_deg",
        "invalid_longitude_deg",
        "nonfinite_longitude_deg",
        "longitude_out_of_range",
    ]

    # Keep optional metadata problems as flags. A blank site closure date is allowed.
    optional_flag_names = [
        name
        for name in checks
        if name not in exclusion_names
        and name != "missing_site_closed_date"
    ]

    def matching_checks(names):
        """Return the names of the selected checks that apply to each row."""
        return F.filter(
            F.array(
                *[
                    F.when(
                        F.coalesce(checks[name], F.lit(False)),
                        F.lit(name),
                    )
                    for name in names
                ]
            ),
            lambda reason: reason.isNotNull(),
        )

    # Keep every check result in a map so the audit can explain the decision.
    diagnostic_entries = []
    for name, condition in checks.items():
        diagnostic_entries.extend(
            [
                F.lit(name),
                F.coalesce(condition, F.lit(False)),
            ]
        )

    # Add exclusion reasons, eligibility and retained quality flags without removing rows.
    return (
        records
        .withColumn(
            "reference_field_flags",
            F.create_map(*diagnostic_entries),
        )
        .withColumn(
            "field_reasons",
            matching_checks(exclusion_names),
        )
        .withColumn(
            "field_eligible",
            F.size("field_reasons") == 0,
        )
        .withColumn(
            "field_disposition",
            F.when(
                F.col("field_eligible"),
                F.lit("eligible"),
            ).otherwise(F.element_at("field_reasons", 1)),
        )
        .withColumn(
            "reference_quality_flags",
            matching_checks(optional_flag_names),
        )
    )

### 7.4 Resolve duplicate identifiers

We count exact copies together. If different records share the same site, monitor or station identifier within a source version, we keep the conflict in the audit.

The check is within one source file and version. The helper temporarily represents diagnostic maps as sorted entries because those entries can be grouped when finding exact copies.

In [28]:
# Reuse the source duplicate rules for site, monitor and station identifiers.
def resolve_reference_keys(records, key_columns):
    """Count identical copies and flag conflicting rows for the same reference identifier."""
    key_columns = list(key_columns)
    if not key_columns:
        raise ValueError("Reference keys must contain at least one column.")

    # Only test key conflicts when all identifier fields are present.
    key_is_usable = F.lit(True)
    for column in key_columns:
        key_is_usable = key_is_usable & ~missing_string(column)

    # Spark cannot group by map columns. Convert each map to sorted key/value entries
    # so maps with the same contents compare equally regardless of entry order.
    groupable = (
        records
        .withColumn(
            "__reference_field_entries",
            F.sort_array(F.map_entries("reference_field_flags")),
        )
        .drop("reference_field_flags")
    )

    # Resolve copies and conflicts, then restore the diagnostic map for the output audit.
    return (
        resolve_source_keys(
            groupable,
            key_columns=key_columns,
            key_is_usable=key_is_usable,
        )
        .withColumn(
            "reference_field_flags",
            F.map_from_entries("__reference_field_entries"),
        )
        .drop("__reference_field_entries")
    )

### 7.5 Build the reference tables and audit

The three reference tables contain the eligible records. Their shared audit keeps the original fields, quality flags and outcome for each selected record.

In [29]:
# Define identifiers and readable output names for the three reference tables.
reference_output_specs = {
    "epa_sites": {
        "id_column": "site_id",
        "keys": {
            "State Code": "state_code", "County Code": "county_code",
            "Site Number": "site_number",
        },
        "text": {
            "Local Site Name": "site_name", "State Name": "state_name",
            "County Name": "county_name", "Land Use": "land_use",
            "Location Setting": "location_setting", "Datum": "datum",
        },
    },
    "epa_monitors": {
        "id_column": "monitor_id",
        "keys": {
            "State Code": "state_code", "County Code": "county_code",
            "Site Number": "site_number", "Parameter Code": "parameter_code",
            "POC": "poc",
        },
        "text": {
            "Parameter Name": "parameter_name", "Monitor Type": "monitor_type",
            "Measurement Scale": "measurement_scale", "Datum": "datum",
            "Last Method Code": "last_method_code", "Last Method": "last_method",
        },
    },
    "noaa_stations": {
        "id_column": "station_id",
        "keys": {"GHCN_ID": "station_id"},
        "text": {
            "NAME": "station_name", "STATE": "state_abbreviation",
            "ICAO": "icao_code", "ISO_CODE": "country_code",
        },
    },
}

# Carry the source file and original-row identity into both outputs.
reference_lineage_columns = [
    "source_provider", "source_dataset", "source_version_sha256",
    "source_file_path", "source_file_sha256", "source_manifest_path",
    "source_record_hash_scheme", "source_record_sha256",
]


def build_reference_outputs(records, reference_name, source_columns):
    """Build an eligible reference table and a counted source audit."""
    # Build the entity ID only when all of its source identifiers are usable.
    spec = reference_output_specs[reference_name]
    usable_key = F.lit(True)
    for column in spec["keys"]:
        usable_key = usable_key & ~missing_string(column)

    entity_id = F.when(
        usable_key,
        F.concat_ws("-", *[F.col(column) for column in spec["keys"]]),
    )

    # Rename identifier columns and add the combined site, monitor or station ID.
    identifiers = [
        F.col(original).alias(output)
        for original, output in spec["keys"].items()
        if output != spec["id_column"]
    ]
    identifiers.append(entity_id.alias(spec["id_column"]))

    # Keep required coordinates; replace unusable optional typed values with null.
    typed_fields = []
    for column in reference_type_specs[reference_name]:
        value = F.col(column)
        if column not in {"latitude_deg", "longitude_deg"}:
            unusable = F.lit(False)
            for prefix in ("missing_", "invalid_", "nonfinite_"):
                unusable = unusable | F.coalesce(
                    F.col("reference_field_flags")[prefix + column],
                    F.lit(False),
                )
            if column == "first_year_of_data":
                unusable = unusable | F.coalesce(
                    F.col("reference_field_flags")["first_year_out_of_range"],
                    F.lit(False),
                )
            value = F.when(~unusable, value)
        typed_fields.append(value.alias(column))

    # Trim optional text and use null when a field is blank or absent.
    text_fields = []
    for original, output in spec["text"].items():
        value = (
            F.when(~missing_string(original), F.trim(F.col(original)))
            if original in records.columns
            else F.lit(None).cast("string")
        )
        text_fields.append(value.alias(output))

    # Build the eligible reference table with flags and source details.
    reference = records.filter("record_eligible").select(
        *identifiers,
        *typed_fields,
        *text_fields,
        F.col("reference_quality_flags").alias("quality_flags"),
        "source_occurrences",
        *reference_lineage_columns,
    )

    # Keep raw fields and check results for eligible and excluded reference records.
    audit_base = (
        records
        .withColumn("reference", F.lit(reference_name))
        .withColumn("entity_id", entity_id)
        .withColumn("raw_fields", named_column_map(source_columns))
        .withColumn("field_flags", F.col("reference_field_flags"))
    )
    common = [
        "reference", "entity_id", "raw_fields", "field_flags",
        "field_disposition", "field_reasons", "reference_quality_flags",
        "key_variant_count", "key_conflict", *reference_lineage_columns,
    ]
    # Count each distinct source record once in the audit.
    representatives = audit_base.select(
        *common,
        F.col("record_disposition").alias("disposition"),
        F.col("record_reasons").alias("reasons"),
        F.lit(1).cast("long").alias("record_count"),
    )
    # Add the extra copies separately so every selected input record is accounted for.
    duplicate_copies = audit_base.filter("duplicate_extra_records > 0").select(
        *common,
        F.lit("exact_duplicate").alias("disposition"),
        F.concat(
            F.array(F.lit("exact_duplicate")), F.col("record_reasons"),
        ).alias("reasons"),
        F.col("duplicate_extra_records").alias("record_count"),
    )

    # Return the eligible reference table and its complete counted audit.
    return {
        "reference": reference,
        "audit": representatives.unionByName(duplicate_copies),
    }

### 7.6 Check the reference totals

We check that input records equal distinct records plus extra copies. The distinct records must all be accounted for as eligible, conflicting or excluded by the field checks. Finally, the eligible audit count must match the reference output count.

The summary below comes from the same resolved records we publish.

In [30]:
# Apply one shared processing flow to the selected rows of a reference type.
def process_reference_records(records, source, reference_name, environment_root):
    """Process selected raw reference records and reconcile their audit."""
    # Require the selected input and its manifest to describe the expected compatible source.
    expected_provider, expected_dataset = reference_name.split("_", 1)
    manifest = json.loads(
        Path(source["manifest_path"]).read_text(encoding="utf-8")
    )
    if (
        source["request"].get("source") != expected_provider
        or source["request"].get("dataset") != expected_dataset
        or manifest["source"].get("provider") != expected_provider
        or manifest["source"].get("dataset") != expected_dataset
        or source["schema"]["status"] != "compatible"
        or manifest["schema"]["status"] != "compatible"
    ):
        raise ValueError(f"{reference_name}: incompatible Bronze source.")

    # Remember the selected input count, then type, classify and trace the records to Bronze.
    input_records = records.count()
    source_columns = records.columns
    typed = type_reference_fields(records, reference_name)
    classified = classify_reference_fields(typed, reference_name)
    with_lineage = attach_bronze_lineage(
        classified, source, environment_root, source_columns,
    )
    # Resolve duplicates once; the counts and both outputs use this same result.
    resolved = resolve_reference_keys(
        with_lineage,
        key_columns=list(reference_output_specs[reference_name]["keys"]),
    )
    # Count original occurrences, distinct rows, extra copies and the three distinct-row outcomes.
    counts = resolved.agg(
        F.coalesce(F.sum("source_occurrences"), F.lit(0)).alias(
            "input_records"
        ),
        F.count("*").alias("distinct_source_rows"),
        F.coalesce(F.sum("duplicate_extra_records"), F.lit(0)).alias(
            "duplicate_extra_records"
        ),
        F.count(F.when(F.col("record_eligible"), 1)).alias(
            "eligible_records"
        ),
        F.count(F.when(F.col("key_conflict"), 1)).alias(
            "quarantined_records"
        ),
        F.count(
            F.when(
                ~F.col("key_conflict") & ~F.col("record_eligible"),
                1,
            )
        ).alias("field_excluded_records"),
    ).first().asDict()

    # Check that no selected input disappeared or was counted twice.
    # Distinct rows must split into eligible, conflicting and field-excluded records.
    if (
        counts["input_records"] != input_records
        or counts["input_records"] != (
            counts["distinct_source_rows"] + counts["duplicate_extra_records"]
        )
        or counts["distinct_source_rows"] != (
            counts["eligible_records"]
            + counts["quarantined_records"]
            + counts["field_excluded_records"]
        )
    ):
        raise ValueError(f"{reference_name}: reference counts do not reconcile.")

    # Build the reference and audit from the resolved records whose totals we just checked.
    result = build_reference_outputs(resolved, reference_name, source_columns)

    # Count output rows and retained flags, then total the input records represented by the audit.
    output_summary = result["reference"].agg(
        F.count("*").alias("output_records"),
        F.count(F.when(F.size("quality_flags") > 0, 1)).alias("flagged_records"),
    ).first().asDict()
    dispositions = {
        row["disposition"]: row["source_records"]
        for row in result["audit"].groupBy("disposition").agg(
            F.sum("record_count").alias("source_records")
        ).collect()
    }
    # Require the audit to explain every input and its eligible total to match the reference table.
    if sum(dispositions.values()) != input_records:
        raise ValueError(f"{reference_name}: audit does not account for all inputs.")
    if dispositions.get("eligible", 0) != output_summary["output_records"]:
        raise ValueError(f"{reference_name}: eligible audit and output counts differ.")

    # Attach a small summary to the two Spark tables.
    result["summary"] = {
        "input_records": input_records,
        **output_summary,
        "dispositions": dispositions,
    }
    return result


# Run the reference processor for all three selected reference types.
reference_results = {}
reference_tables = {}
reference_audits = {}
for reference_name, _, _, _ in reference_specs:
    result = process_reference_records(
        records=reference_profile_rows[reference_name],
        source=reference_sources[reference_name],
        reference_name=reference_name,
        environment_root=silver_context.environment_root,
    )
    reference_results[reference_name] = result
    reference_tables[reference_name] = result["reference"]
    reference_audits[reference_name] = result["audit"]
    summary = result["summary"]

    audited = sum(summary["dispositions"].values())
    print(f"{reference_name}: {summary['input_records']} selected, "
          f"{summary['output_records']} eligible ({summary['flagged_records']} flagged), "
          f"{audited} accounted for: {summary['dispositions']}")

# Optionally display the selected locations and their retained flags.
if show_diagnostics:
    for reference_name, reference_table in reference_tables.items():
        print(reference_name)
        reference_table.select(
            reference_output_specs[reference_name]["id_column"],
            *reference_type_specs[reference_name],
            "quality_flags",
        ).show(truncate=False, vertical=True)

epa_sites: 1 selected, 1 eligible (0 flagged), 1 accounted for: {'eligible': 1}


epa_monitors: 1 selected, 1 eligible (0 flagged), 1 accounted for: {'eligible': 1}
noaa_stations: 1 selected, 1 eligible (0 flagged), 1 accounted for: {'eligible': 1}


## 8. Publish source Silver

### 8.1 Define the eight output tables

We have two measurement tables, three reference tables and three audits. For each table we specify its columns, required values and the key that identifies a row. Temperature still uses the original report timestamp.

A **contract** describes the table we expect. **Staging** holds files that are not yet published. A **manifest** identifies a complete published release and its files. A **completion report** records that this run successfully reopened the release.

The sequence below is: define the contracts, stage the files, read them back and validate them, publish the bundle, then reopen the published release.

In [31]:
# Define the expected columns and validation rules for the eight source Silver tables.
def silver_column_types(data_type, names):
    """Return a column-to-type mapping for fields that share one Spark type."""
    return {name: data_type for name in names.split()}


# Share the source-tracing fields across observations, references and audits.
silver_lineage_types = silver_column_types(
    "string",
    "source_provider source_dataset source_version_sha256 source_file_path "
    "source_file_sha256 source_manifest_path source_record_hash_scheme "
    "source_record_sha256",
)

# Audits also carry original values, check results, outcomes and represented record counts.
silver_audit_types = {
    **silver_lineage_types,
    **silver_column_types("string", "field_disposition disposition"),
    **silver_column_types("array<string>", "field_reasons reasons"),
    "raw_fields": "map<string,string>",
    "field_flags": "map<string,boolean>",
    "key_variant_count": "bigint",
    "key_conflict": "boolean",
    "record_count": "bigint",
}

# Define measurement and audit types explicitly so accidental schema changes are caught.
source_silver_schemas = {
    "ozone_observations": {
        **silver_lineage_types,
        **silver_column_types(
            "string",
            "state_code county_code site_number parameter_code poc monitor_id "
            "ozone_raw source_unit mdl_raw qualifier_code qualifier_raw "
            "method_type method_code method_name",
        ),
        "timestamp_utc": "timestamp",
        "ozone_ppm": "double",
        "mdl_ppm": "double",
        "quality_flags": "array<string>",
        "source_occurrences": "bigint",
    },
    "temperature_reports": {
        **silver_lineage_types,
        **silver_column_types(
            "string",
            "station_id temperature_raw timestamp_raw source_code quality_code "
            "report_type measurement_code source_station_id source_code_raw "
            "quality_code_raw report_type_raw measurement_code_raw "
            "source_station_id_raw",
        ),
        "timestamp_utc": "timestamp",
        "temperature_c": "double",
        "quality_flags": "array<string>",
        "source_occurrences": "bigint",
        "supporting_source_records": "bigint",
    },
    "ozone_audit": {
        **silver_audit_types,
        "monitor_id": "string",
        "timestamp_utc": "timestamp",
    },
    "temperature_audit": {
        **silver_audit_types,
        "station_id": "string",
        "timestamp_utc": "timestamp",
        "key_variant_count": "int",
        "key_source_records": "bigint",
        "redundant_temperature": "boolean",
    },
    "reference_audit": {
        **silver_audit_types,
        "reference": "string",
        "entity_id": "string",
        "reference_quality_flags": "array<string>",
    },
}

# Build the three reference schemas from the field definitions in section 7.
for reference_name, spec in reference_output_specs.items():
    source_silver_schemas[reference_name] = {
        **silver_lineage_types,
        **{name: "string" for name in spec["keys"].values()},
        spec["id_column"]: "string",
        **{
            name: data_type
            for name, (_, data_type) in reference_type_specs[reference_name].items()
        },
        **{name: "string" for name in spec["text"].values()},
        "quality_flags": "array<string>",
        "source_occurrences": "bigint",
    }

# Combine the three reference audits, keeping the reference type on each audit row.
reference_audit_combined = reference_audits["epa_sites"].unionByName(
    reference_audits["epa_monitors"]
).unionByName(reference_audits["noaa_stations"])

# Collect two measurement tables, three reference tables and three audits.
source_silver_tables = {
    "ozone_observations": ozone_observations,
    "temperature_reports": temperature_observations,
    **reference_tables,
    "ozone_audit": ozone_audit,
    "temperature_audit": temperature_audit,
    "reference_audit": reference_audit_combined,
}

# Record the cleaning-rule versions used to produce this bundle.
source_silver_rule_versions = {
    "ozone": "ozone-source-v1",
    "temperature": "temperature-source-v1",
    "references": "reference-source-v1",
}


def source_silver_contract(name, key, required, expected_rows=None):
    """Return the expected column types, unique key, required fields and optional row count."""
    return {
        "schema_version": f"{name}-v1",
        "columns": source_silver_schemas[name],
        "key": key,
        "required": list(dict.fromkeys([*key, *required, *silver_lineage_types])),
        "expected_rows": expected_rows,
    }


# Set each table's unique key, non-null fields and expected row count where known.
source_silver_contracts = {
    "ozone_observations": source_silver_contract(
        "ozone_observations",
        ["monitor_id", "timestamp_utc"],
        "state_code county_code site_number parameter_code poc ozone_ppm "
        "source_unit quality_flags source_occurrences".split(),
        ozone_result_summary["after"]["records"],
    ),
    "temperature_reports": source_silver_contract(
        "temperature_reports",
        ["station_id", "timestamp_utc"],
        "temperature_c source_code quality_code report_type quality_flags "
        "source_occurrences supporting_source_records".split(),
        temperature_result_summary["after"]["records"],
    ),
}

# Require exactly one eligible reference row for each selected profile identifier.
for reference_name, spec in reference_output_specs.items():
    source_silver_contracts[reference_name] = source_silver_contract(
        reference_name,
        [spec["id_column"]],
        [
            *spec["keys"].values(), "latitude_deg", "longitude_deg",
            "quality_flags", "source_occurrences",
        ],
        expected_rows=1,
    )

# Audit keys include source identity and disposition so extra-copy entries remain distinct.
silver_audit_key = [
    "source_provider", "source_dataset", "source_version_sha256",
    "source_file_path", "source_file_sha256", "source_record_sha256",
    "disposition",
]
for dataset in ("ozone_audit", "temperature_audit", "reference_audit"):
    required = [
        "raw_fields", "field_flags", "field_disposition", "field_reasons",
        "key_conflict", "reasons", "record_count",
    ]
    if dataset == "reference_audit":
        required += ["reference", "reference_quality_flags"]
    source_silver_contracts[dataset] = source_silver_contract(
        dataset, silver_audit_key.copy(), required,
    )

# Check all eight in-memory schemas against their contracts before writing files.
if set(source_silver_tables) != set(source_silver_contracts):
    raise ValueError("Silver tables and contracts do not match.")

for dataset, table in source_silver_tables.items():
    expected = source_silver_contracts[dataset]["columns"]
    if len(table.columns) != len(set(table.columns)) or dict(table.dtypes) != expected:
        raise ValueError(
            f"{dataset}: schema differs from its contract. "
            f"Expected {expected}; received {dict(table.dtypes)}"
        )

print(f"Source Silver schemas checked: {len(source_silver_tables)} datasets.")

Source Silver schemas checked: 8 datasets.


### 8.2 Write the Parquet files

We write the tables to a staging directory for this run. These regional outputs are small, so we use one write partition per table and record the file sizes, checksums and row counts.

In [32]:
# Write an unpublished bundle and return the details needed for read-back validation.
def stage_source_silver(tables, contracts, context):
    """Write a new unpublished Parquet bundle and record its file inventory."""
    if set(tables) != set(contracts):
        raise ValueError("Tables and contracts must have the same dataset names.")

    # Keep staging inside this run's directory and check the table schemas before writing.
    root = Path(context.environment_root).resolve()
    parent = Path(context.staging_root).resolve()
    if parent == root / "staging" or not parent.is_relative_to(root / "staging"):
        raise ValueError("Staging must be inside this environment's run directory.")

    for name, table in tables.items():
        if (
            not name.isidentifier()
            or len(table.columns) != len(set(table.columns))
            or dict(table.dtypes) != contracts[name]["columns"]
        ):
            raise ValueError(f"{name}: invalid dataset name or schema.")

    # Create a new staging folder so a previous attempt is never silently overwritten.
    stage_path = parent / f"source-silver-{time.time_ns()}"
    stage_path.mkdir(parents=True, exist_ok=False)
    started = time.perf_counter()
    created_at_utc = datetime.now(timezone.utc).isoformat()
    datasets = {}

    # Write each small regional table and record its row count.
    for name, table in tables.items():
        destination = stage_path / name
        # Counting and writing both run Spark jobs. Cache this table to reuse the result,
        # and release it afterwards only if this function created the cache.
        owned_cache = not table.is_cached
        if owned_cache:
            table.cache()
        try:
            row_count = table.count()
            # One write partition is enough for these small outputs; preserve their schema in Parquet.
            table.coalesce(1).write.parquet(
                str(destination), mode="errorifexists", compression="snappy",
            )
        finally:
            if owned_cache:
                table.unpersist(blocking=True)

        # Record each Parquet file's relative path, size and checksum after the write.
        files = [
            {
                "path": path.relative_to(stage_path).as_posix(),
                "bytes": path.stat().st_size,
                "sha256": file_sha256(path),
            }
            for path in sorted(destination.glob("*.parquet"))
        ]
        if not files:
            raise ValueError(f"{name}: no Parquet files were produced.")

        datasets[name] = {
            "path": name,
            "row_count": row_count,
            "schema": table.schema.jsonValue(),
            "files": files,
            "bytes": sum(item["bytes"] for item in files),
        }
        print(f"Staged {name}: {row_count} rows, {len(files)} Parquet file(s).")

    # Hash the contracts and return the staged bundle description, not a publication.
    contract_json = json.dumps(contracts, sort_keys=True, separators=(",", ":"))
    return {
        "status": "staged",
        "created_at_utc": created_at_utc,
        "run_id": context.run_id,
        "environment": context.environment,
        "environment_root": str(root),
        "bundle_path": stage_path.relative_to(root).as_posix(),
        "format": "parquet",
        "compression": "snappy",
        "write_partitions": 1,
        "partition_columns": [],
        "contract_sha256": hashlib.sha256(contract_json.encode("utf-8")).hexdigest(),
        "datasets": datasets,
        "elapsed_seconds": round(time.perf_counter() - started, 3),
    }


# Stage the actual source Silver tables now, then display the saved file inventory.
source_silver_stage = None
source_silver_stage = stage_source_silver(
    source_silver_tables, source_silver_contracts, silver_context,
)

print("Status:", source_silver_stage["status"])
print("Staging directory:", (
    silver_context.environment_root / source_silver_stage["bundle_path"]
))
print("Staging time (seconds):", source_silver_stage["elapsed_seconds"])

spark.createDataFrame(
    [
        (name, info["row_count"], len(info["files"]), info["bytes"])
        for name, info in source_silver_stage["datasets"].items()
    ],
    "dataset STRING, records LONG, parquet_files INT, parquet_bytes LONG",
).orderBy("dataset").show(truncate=False)

Staged ozone_observations: 5071 rows, 1 Parquet file(s).


Staged temperature_reports: 11686 rows, 1 Parquet file(s).
Staged epa_sites: 1 rows, 1 Parquet file(s).


Staged epa_monitors: 1 rows, 1 Parquet file(s).
Staged noaa_stations: 1 rows, 1 Parquet file(s).


Staged ozone_audit: 5166 rows, 1 Parquet file(s).


26/11/13 23:16:08 WARN DAGScheduler: Broadcasting large task binary with size 1038.9 KiB


Staged temperature_audit: 11851 rows, 1 Parquet file(s).


26/11/13 23:16:13 WARN DAGScheduler: Broadcasting large task binary with size 1057.9 KiB


Staged reference_audit: 3 rows, 1 Parquet file(s).
Status: staged
Staging directory: /home/ubuntu/dat535-final-project-data/dev/staging/bc50a11d476d4e0083d429448023dcef/source-silver-1794611703501764741
Staging time (seconds): 70.069
+-------------------+-------+-------------+-------------+
|dataset            |records|parquet_files|parquet_bytes|
+-------------------+-------+-------------+-------------+
|epa_monitors       |1      |1            |10524        |
|epa_sites          |1      |1            |9970         |
|noaa_stations      |1      |1            |8094         |
|ozone_audit        |5166   |1            |444616       |
|ozone_observations |5071   |1            |385139       |
|reference_audit    |3      |1            |11385        |
|temperature_audit  |11851  |1            |3128825      |
|temperature_reports|11686  |1            |944457       |
+-------------------+-------+-------------+-------------+



### 8.3 Read back and check the saved tables

We check the stored files, schemas, counts and keys. Each retained observation must match an eligible audit entry and point back to its selected Bronze source.

The validator checks file contents, table structure and audit accounting. For source Silver, it also compares the original-row identities of observations and eligible audit entries. The full hourly row comparison happens separately in 9.4b.

In [33]:
# Define the source-identity lookup and the full saved-table validator.
def selected_bronze_lineage(source, environment_root):
    """Read the expected file identity from the selected Bronze manifest."""
    root = Path(environment_root).resolve()
    manifest_path = Path(source["manifest_path"]).resolve(strict=True)
    data_path = Path(source["data_path"]).resolve(strict=True)
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest["manifest_version"] != 1 or manifest["schema"]["status"] != "compatible":
        raise ValueError("The selected Bronze manifest is incompatible.")

    # Find the exact data file in the selected Bronze manifest.
    relative_data_path = data_path.relative_to(root).as_posix()
    matches = [
        item for item in [manifest["original_file"], *manifest["extracted_files"]]
        if item["path"] == relative_data_path
    ]
    if len(matches) != 1:
        raise ValueError("The selected Bronze file must match one manifest entry.")

    return {
        "source_provider": manifest["source"]["provider"],
        "source_dataset": manifest["source"]["dataset"],
        "source_version_sha256": manifest["original_file"]["sha256"],
        "source_file_path": relative_data_path,
        "source_file_sha256": matches[0]["sha256"],
        "source_manifest_path": manifest_path.relative_to(root).as_posix(),
        "source_record_hash_scheme": "sha256-json-raw-fields-v1",
    }


def validate_source_silver(stage, contracts, context, inputs):
    """Validate staged files, table contracts, lineage and audit reconciliation."""
    # Require this stage, its contracts and the current run to agree.
    root = Path(context.environment_root).resolve()
    if (
        stage["status"] != "staged"
        or Path(stage["environment_root"]).resolve() != root
        or stage["environment"] != context.environment
        or stage["run_id"] != context.run_id
        or stage["contract_sha256"] != silver_json_sha256(contracts)
        or set(stage["datasets"]) != set(contracts)
    ):
        raise ValueError("The staged bundle does not match this run and its contracts.")

    def contained_path(base, relative):
        """Resolve an existing relative path and require it to stay inside the given folder."""
        relative = Path(relative)
        if relative.is_absolute() or ".." in relative.parts:
            raise ValueError(f"Expected a contained relative path: {relative}")
        path = (base / relative).resolve(strict=True)
        if path == base or not path.is_relative_to(base):
            raise ValueError(f"Path is outside the expected directory: {path}")
        return path

    # Resolve the staged bundle without allowing paths outside this run's staging directory.
    bundle = contained_path(root, stage["bundle_path"])
    staging_root = Path(context.staging_root).resolve()
    if bundle == staging_root or not bundle.is_relative_to(staging_root):
        raise ValueError("The bundle is outside this run's staging directory.")

    # Read each saved dataset and collect its validation result.
    readback, checks = {}, {}
    for name, contract in contracts.items():
        info = stage["datasets"][name]
        directory = contained_path(bundle, info["path"])
        # Require the recorded inventory to match all Parquet files in this dataset.
        paths = [contained_path(bundle, item["path"]) for item in info["files"]]
        if (
            not paths
            or len(paths) != len(set(paths))
            or any(not path.is_file() or not path.is_relative_to(directory) for path in paths)
            or set(paths) != {path.resolve() for path in directory.rglob("*.parquet")}
            or info["bytes"] != sum(item["bytes"] for item in info["files"])
        ):
            raise ValueError(f"{name}: the Parquet file inventory does not match.")

        # Compare file sizes and hashes to detect changed or incomplete files before reading.
        for path, item in zip(paths, info["files"]):
            if path.stat().st_size != item["bytes"] or file_sha256(path) != item["sha256"]:
                raise ValueError(f"{name}: file size or checksum mismatch: {path.name}")

        # Read all listed files and verify column order and types against the saved contract.
        table = (
            spark.read
            .option("mergeSchema", "true")
            .option("ignoreCorruptFiles", "false")
            .option("ignoreMissingFiles", "false")
            .parquet(*[str(path) for path in paths])
        )
        stored_columns = [field["name"] for field in info["schema"]["fields"]]
        if table.columns != stored_columns or dict(table.dtypes) != contract["columns"]:
            raise ValueError(f"{name}: persisted column names or types changed.")

        # Build missing-value checks for required fields, including blank required text.
        bad_required = F.lit(False)
        for column in contract["required"]:
            missing = F.col(column).isNull()
            if contract["columns"][column] == "string":
                missing = missing | (F.trim(F.col(column)) == "")
            bad_required = bad_required | missing

        # Check finite numbers, array contents, hash formats and valid positive counts.
        bad_numeric, bad_array, bad_hash, bad_count = (F.lit(False) for _ in range(4))
        for column, data_type in contract["columns"].items():
            value = F.col(column)
            if data_type in {"float", "double"}:
                bad_numeric = bad_numeric | (
                    value.isNotNull() & ~finite_number(column)
                )
            if data_type == "array<string>":
                bad_array = bad_array | F.coalesce(
                    F.exists(value, lambda item: item.isNull()), F.lit(False),
                )
            if column.endswith("_sha256"):
                bad_hash = bad_hash | ~F.coalesce(
                    value.rlike("^[0-9a-f]{64}$"), F.lit(False),
                )
        for column in (
            "source_occurrences", "supporting_source_records",
            "record_count", "key_variant_count", "key_source_records",
        ):
            if column in table.columns:
                bad_count = bad_count | (
                    F.col(column).isNotNull() & (F.col(column) < 1)
                )
        if "record_count" in table.columns:
            bad_count = bad_count | (
                (F.col("disposition") != "exact_duplicate")
                & (F.col("record_count") != 1)
            )

        failures = {
            "missing_required": bad_required, "nonfinite_numbers": bad_numeric,
            "null_array_items": bad_array, "invalid_hashes": bad_hash,
            "invalid_counts": bad_count,
        }
        # Make Spark read every column, including nested audit values.
        # Counting rows alone would not require Spark to decode all those fields.
        decoded = F.to_json(
            F.struct(*[F.col(column) for column in table.columns]),
            options={"ignoreNullFields": "false"},
        )
        measured = table.agg(
            F.count("*").alias("records"),
            F.sum(F.length(decoded)).alias("decoded_characters"),
            *[
                F.count(F.when(condition, 1)).alias(label)
                for label, condition in failures.items()
            ],
        ).first().asDict()
        if any(measured[label] for label in failures):
            raise ValueError(f"{name}: persisted value checks failed: {measured}")
        if (
            measured["records"] != info["row_count"]
            or (
                contract["expected_rows"] is not None
                and measured["records"] != contract["expected_rows"]
            )
        ):
            raise ValueError(f"{name}: persisted row count does not match.")

        # Require each persisted table key to identify at most one row.
        if table.groupBy(*contract["key"]).count().filter("count > 1").limit(1).count():
            raise ValueError(f"{name}: duplicate output keys.")

        readback[name] = table
        checks[name] = {
            "records": measured["records"], "parquet_files": len(paths),
            "status": "passed",
        }

    # Ensure every saved table is covered as either a clean output or its audit.
    audit_names = {item["audit"] for item in inputs.values()}
    if (set(inputs) & audit_names) or (set(inputs) | audit_names) != set(readback):
        raise ValueError("Every persisted dataset must have one validation role.")

    audit_checks, accounted_audit_rows = {}, Counter()
    lineage_columns = [*silver_lineage_types]
    # Reconcile each clean table with its saved audit and selected Bronze source.
    for name, expected in inputs.items():
        clean = readback[name]
        audit_name = expected["audit"]
        audit = readback[audit_name]
        if "reference" in expected:
            audit = audit.filter(F.col("reference") == F.lit(expected["reference"]))

        # Compare represented input counts by outcome, not just the number of audit entries.
        audit_rows = audit.groupBy("disposition").agg(
            F.sum("record_count").alias("source_records"),
            F.count("*").alias("audit_rows"),
        ).collect()
        dispositions = {row["disposition"]: row["source_records"] for row in audit_rows}
        accounted_audit_rows[audit_name] += sum(row["audit_rows"] for row in audit_rows)
        if (
            dispositions != expected["dispositions"]
            or sum(dispositions.values()) != expected["input_records"]
            or dispositions.get("eligible", 0) != checks[name]["records"]
        ):
            raise ValueError(f"{name}: persisted audit counts do not reconcile.")

        # Check that retained quality flags survived the Parquet round trip.
        if "quality_flags" in expected:
            flags = {
                row["quality_flag"]: row["count"]
                for row in clean.select(
                    F.explode("quality_flags").alias("quality_flag")
                ).groupBy("quality_flag").count().collect()
            }
            if flags != expected["quality_flags"]:
                raise ValueError(f"{name}: persisted quality-flag counts differ.")
        if "flagged_records" in expected:
            if clean.filter(F.size("quality_flags") > 0).count() != expected["flagged_records"]:
                raise ValueError(f"{name}: persisted flagged-record count differs.")

        # Require clean and audit rows to point to the selected Bronze file and version.
        # eqNullSafe treats two nulls as equal and a null versus a value as different.
        lineage = selected_bronze_lineage(expected["source"], root)
        mismatch = F.lit(False)
        for column, value in lineage.items():
            mismatch = mismatch | ~F.col(column).eqNullSafe(F.lit(value))
        combined_lineage = clean.select(*lineage).unionByName(audit.select(*lineage))
        if combined_lineage.filter(mismatch).limit(1).count():
            raise ValueError(f"{name}: lineage differs from the selected Bronze source.")

        # Find missing and extra source-row identities in both directions.
        # exceptAll preserves copy counts, so repeated rows cannot hide a mismatch.
        accepted = audit.filter(F.col("disposition") == "eligible").select(*lineage_columns)
        observed = clean.select(*lineage_columns)
        differences = observed.exceptAll(accepted).unionByName(accepted.exceptAll(observed))
        if differences.limit(1).count():
            raise ValueError(f"{name}: output and eligible-audit lineage differ.")

        audit_checks[name] = {
            "input_records": expected["input_records"],
            "dispositions": dispositions,
            "lineage": lineage,
            "status": "passed",
        }

    # Check that the validation above covered every row of each audit exactly once.
    if any(accounted_audit_rows[name] != checks[name]["records"] for name in audit_names):
        raise ValueError("Some audit rows are unaccounted for or counted more than once.")

    # Return validation evidence tied to this exact stage and the reloaded Spark tables.
    report = {
        "status": "validated",
        "validated_at_utc": datetime.now(timezone.utc).isoformat(),
        "run_id": context.run_id,
        "bundle_path": stage["bundle_path"],
        "stage_sha256": silver_json_sha256(stage),
        "contract_sha256": stage["contract_sha256"],
        "datasets": checks,
        "audits": audit_checks,
    }
    return report, readback


# Supply the expected input counts, audit outcomes and flags from the processing summaries.
source_silver_validation_inputs = {
    "ozone_observations": {
        "source": ozone_source, "audit": "ozone_audit",
        "input_records": ozone_result_summary["before"]["records"],
        "dispositions": ozone_result_summary["dispositions"],
        "quality_flags": ozone_result_summary["quality_flags"],
    },
    "temperature_reports": {
        "source": temperature_source, "audit": "temperature_audit",
        "input_records": temperature_result_summary["before"]["records"],
        "dispositions": temperature_result_summary["dispositions"],
        "quality_flags": temperature_result_summary["quality_flags"],
    },
}
for reference_name, result in reference_results.items():
    source_silver_validation_inputs[reference_name] = {
        "source": reference_sources[reference_name],
        "audit": "reference_audit", "reference": reference_name,
        "input_records": result["summary"]["input_records"],
        "dispositions": result["summary"]["dispositions"],
        "flagged_records": result["summary"]["flagged_records"],
    }

# Validate the actual staged bundle now and display its per-table results.
source_silver_validation = None
source_silver_readback = None
source_silver_validation, source_silver_readback = validate_source_silver(
    source_silver_stage, source_silver_contracts, silver_context,
    source_silver_validation_inputs,
)

spark.createDataFrame(
    [
        (name, result["records"], result["parquet_files"], result["status"])
        for name, result in source_silver_validation["datasets"].items()
    ],
    "dataset STRING, records LONG, parquet_files INT, validation STRING",
).orderBy("dataset").show(truncate=False)

print("Audit and lineage reconciliations passed:", len(source_silver_validation["audits"]))
print("Validation status:", source_silver_validation["status"])
print("Publication status: unpublished")

+-------------------+-------+-------------+----------+
|dataset            |records|parquet_files|validation|
+-------------------+-------+-------------+----------+
|epa_monitors       |1      |1            |passed    |
|epa_sites          |1      |1            |passed    |
|noaa_stations      |1      |1            |passed    |
|ozone_audit        |5166   |1            |passed    |
|ozone_observations |5071   |1            |passed    |
|reference_audit    |3      |1            |passed    |
|temperature_audit  |11851  |1            |passed    |
|temperature_reports|11686  |1            |passed    |
+-------------------+-------+-------------+----------+

Audit and lineage reconciliations passed: 5
Validation status: validated
Publication status: unpublished


### 8.4 Define how a release is published

Once validation passes, the publisher moves the files to a versioned directory and writes the manifest last. Later stages use that manifest to identify a complete release.

This cell defines the publisher without calling it. Section 8.5 exercises it on isolated fixture copies, and 8.6 publishes the actual bundle. Reusing an identical publication attempt is different from rerunning the whole notebook, which creates new run metadata.

In [34]:
# Define how a fully validated source Silver bundle becomes a published release.
def publish_source_silver(
    stage, validation, contracts, context, *, profile, rule_versions, provenance,
):
    """Move validated source tables to their release directory and write the manifest last."""
    # Make an independent JSON copy so the manifest records the supplied metadata consistently.
    stage, validation, contracts, profile, rule_versions, provenance = json.loads(
        json.dumps(
            [stage, validation, contracts, profile, rule_versions, provenance],
            allow_nan=False,
        )
    )
    root = Path(context.environment_root).resolve()
    stage_hash = silver_json_sha256(stage)
    contract_hash = silver_json_sha256(contracts)
    if (
        stage["status"] != "staged"
        or Path(stage["environment_root"]).resolve() != root
        or stage["environment"] != context.environment
        or stage["run_id"] != context.run_id
        or stage["contract_sha256"] != contract_hash
        or validation["status"] != "validated"
        or validation["run_id"] != context.run_id
        or validation["bundle_path"] != stage["bundle_path"]
        or validation["stage_sha256"] != stage_hash
        or validation["contract_sha256"] != contract_hash
        or set(stage["datasets"]) != set(contracts)
        or set(validation["datasets"]) != set(contracts)
    ):
        raise ValueError(
            "Publication requires a matching stage, validation and contracts."
        )

    # Require a passed file and count check for every table.
    for name, info in stage["datasets"].items():
        check = validation["datasets"][name]
        if (
            check["status"] != "passed"
            or check["records"] != info["row_count"]
            or check["parquet_files"] != len(info["files"])
        ):
            raise ValueError(
                f"{name}: validation is incomplete or inconsistent."
            )

    # All five clean tables must also have passed their audit and source-tracing checks.
    clean_names = {
        "ozone_observations", "temperature_reports",
        "epa_sites", "epa_monitors", "noaa_stations",
    }
    if (
        set(validation["audits"]) != clean_names
        or any(
            item["status"] != "passed"
            for item in validation["audits"].values()
        )
    ):
        raise ValueError(
            "All five audit and lineage reconciliations must pass."
        )

    # Use the profile region and year in the release path, rejecting unsafe path components.
    region = profile["region"].strip().lower().replace(" ", "_")
    year = profile["year"]
    if (
        not region
        or any(
            character not in "abcdefghijklmnopqrstuvwxyz0123456789_-"
            for character in region
        )
        or type(year) is not int
        or not 1 <= year <= 9999
        or not rule_versions
        or not provenance
    ):
        raise ValueError(
            "A valid region, year, rule versions and provenance are required."
        )

    # Resolve and check the staging location before moving any files.
    staged_bundle = silver_child_path(root, stage["bundle_path"])
    staging_root = Path(context.staging_root).resolve()
    if (
        not staging_root.is_relative_to(root / "staging")
        or staged_bundle == staging_root
        or not staged_bundle.is_relative_to(staging_root)
    ):
        raise ValueError(
            "The bundle is outside this run's staging directory."
        )

    # Hash the staged bundle description to identify this attempt.
    # It includes run metadata, so a fresh run can get a new ID with the same table values.
    publication_id = stage_hash
    suffix = f"silver/source/{region}/{year}/{publication_id}"
    final_bundle = silver_child_path(root, f"data/{suffix}")
    manifest_path = silver_child_path(
        root, f"published/{suffix}/manifest.json"
    )
    publication_root = silver_child_path(root, "published/silver/source")

    # Save the table contracts, input profile, rule versions and provenance in the manifest.
    identity = {
        "manifest_version": 1,
        "kind": "source_silver",
        "status": "published",
        "publication_id": publication_id,
        "run_id": context.run_id,
        "environment": context.environment,
        "bundle_path": final_bundle.relative_to(root).as_posix(),
        "stage_sha256": stage_hash,
        "contract_sha256": contract_hash,
        "profile": profile,
        "rule_versions": rule_versions,
        "provenance": provenance,
        "contracts": contracts,
        "storage": {
            key: stage[key]
            for key in (
                "format", "compression", "write_partitions",
                "partition_columns",
            )
        },
        "datasets": stage["datasets"],
    }

    # Allow only one source Silver publisher to write in this environment at a time.
    publication_root.mkdir(parents=True, exist_ok=True)
    lock = publication_root / ".publish.lock"
    try:
        lock.mkdir()
    except FileExistsError as error:
        raise RuntimeError(
            "Publication lock already exists; inspect it before retrying: "
            f"{lock}"
        ) from error

    try:
        # An identical retry reuses the release only if its metadata and stored files still match.
        if manifest_path.exists():
            existing = json.loads(
                manifest_path.read_text(encoding="utf-8")
            )
            if {key: existing.get(key) for key in identity} != identity:
                raise ValueError(
                    "The existing publication has different metadata."
                )

            recorded_validation = existing.get("validation", {})
            if any(
                recorded_validation.get(key) != value
                for key, value in validation.items()
                if key != "validated_at_utc"
            ):
                raise ValueError(
                    "The existing publication has different validation results."
                )

            verify_silver_file_inventory(
                final_bundle, existing["datasets"]
            )
            return {
                "outcome": "reused",
                "manifest_path": str(manifest_path),
                "manifest": existing,
            }

        # Data without a manifest is an incomplete attempt; do not overwrite it.
        if final_bundle.exists() or final_bundle.is_symlink():
            raise FileExistsError(
                f"Data exists without a publication manifest: {final_bundle}. "
                "Inspect the incomplete attempt, then stage and validate "
                "a new attempt."
            )

        # Verify the staged files once more before moving them to the release directory.
        verify_silver_file_inventory(
            staged_bundle, stage["datasets"]
        )
        final_bundle.parent.mkdir(parents=True, exist_ok=True)
        manifest_path.parent.mkdir(parents=True, exist_ok=True)
        if staged_bundle.stat().st_dev != final_bundle.parent.stat().st_dev:
            raise ValueError(
                "Staging and final data must be on the same filesystem."
            )

        manifest = {
            **identity,
            "published_at_utc": datetime.now(timezone.utc).isoformat(),
            "validation": validation,
        }

        # Readers may use the bundle only after its manifest exists.
        staged_bundle.rename(final_bundle)
        write_json(manifest_path, manifest)

        return {
            "outcome": "published",
            "manifest_path": str(manifest_path),
            "manifest": manifest,
        }
    # Release the publication lock whether publication succeeds or raises an error.
    finally:
        lock.rmdir()


print("Source Silver publication helper ready. No data published.")

Source Silver publication helper ready. No data published.


### 8.5 Test the cleaning and publication rules

Small test cases in `tests/silver_checks.py` include situations the Alabama files did not contain, such as conflicting readings. We record the results and which test code ran.

The tests call the functions defined in this kernel. Publication tests copy the staged files into temporary fixture folders. They check successful publication, retry behaviour and rejection cases without publishing those copies as the actual source release.

In [35]:
# Identify the saved code and run the small cases that exercise Silver rules.
def notebook_code_sha256(notebook):
    """Hash saved code cells without outputs or execution counts."""
    return silver_json_sha256([
        "".join(cell["source"])
        for cell in notebook["cells"]
        if cell["cell_type"] == "code"
    ])

# Import this checkout's Silver tests and record the saved test-file checksum.
sys.path.insert(0, str(project_dir / "tests"))
import silver_checks

fixture_file = project_dir / "tests" / "silver_checks.py"
if Path(silver_checks.__file__).resolve() != fixture_file.resolve():
    raise RuntimeError("Silver checks must come from this checkout.")
fixture_code_hash = file_sha256(fixture_file)
# Test readers, cleaning, references and publication using the functions defined above.
fixture_results = {
    "readers": silver_checks.verify_delimited_readers(
        staging_root=silver_context.staging_root,
        read_validated_ozone_csv=read_validated_ozone_csv,
        read_validated_delimited_file=read_validated_delimited_file,
    ),
    "ozone": silver_checks.verify_ozone_rules(
        spark=spark,
        type_ozone_fields=type_ozone_fields,
        ozone_field_checks=ozone_field_checks,
        classify_ozone_fields=classify_ozone_fields,
        resolve_source_keys=resolve_source_keys,
        ozone_key_columns=ozone_key_columns,
        build_ozone_observations=build_ozone_observations,
        build_ozone_audit=build_ozone_audit,
    ),
    "temperature": silver_checks.verify_temperature_rules(
        spark=spark,
        profile=alabama_profile,
        temperature_columns=temperature_inspection_columns,
        type_temperature_fields=type_temperature_fields,
        temperature_field_checks=temperature_field_checks,
        classify_temperature_fields=classify_temperature_fields,
        resolve_temperature_keys=resolve_temperature_keys,
        build_temperature_observations=build_temperature_observations,
        build_temperature_audit=build_temperature_audit,
    ),
    "reference_fields": silver_checks.verify_reference_fields(
        spark=spark,
        type_reference_fields=type_reference_fields,
        classify_reference_fields=classify_reference_fields,
    ),
    "reference_outputs": silver_checks.verify_reference_outputs(
        spark=spark,
        type_reference_fields=type_reference_fields,
        classify_reference_fields=classify_reference_fields,
        resolve_reference_keys=resolve_reference_keys,
        build_reference_outputs=build_reference_outputs,
    ),
    # Publication fixtures use isolated copies under staging, not the actual release directory.
    "publication": silver_checks.verify_silver_publication(
        stage=source_silver_stage,
        validation=source_silver_validation,
        contracts=source_silver_contracts,
        context=silver_context,
        profile=alabama_profile,
        rule_versions=source_silver_rule_versions,
        publish_source_silver=publish_source_silver,
        verify_silver_file_inventory=verify_silver_file_inventory,
        silver_json_sha256=silver_json_sha256,
    ),
}
print(json.dumps(fixture_results, indent=2))

# Tie the fixture results to this staged bundle and the test-file contents.
source_silver_checks = {
    "status": "passed",
    "run_id": silver_context.run_id,
    "stage_sha256": silver_json_sha256(source_silver_stage),
    "fixture_file_sha256": fixture_code_hash,
    "results": fixture_results,
}

valid: accepted; decoded values verified
empty_final_value: accepted; decoded values verified
quoted_comma: accepted; decoded values verified
escaped_quotes: accepted; decoded values verified
too_few_fields: rejected; reason verified
too_many_fields: rejected; reason verified
unclosed_quote: rejected; reason verified
Ozone CSV reader fixtures passed: 7 cases.
valid: accepted; decoded values verified
empty_final_value: accepted; decoded values verified
quoted_pipe: accepted; decoded values verified
escaped_quotes: accepted; decoded values verified
too_few_fields: rejected; reason verified
too_many_fields: rejected; reason verified
unclosed_quote: rejected; reason verified
multiline_record: rejected; reason verified
Temperature PSV reader fixtures passed: 8 cases.
Ozone field fixtures passed: 17
Ozone key fixtures passed: 12
Ozone audit fixtures passed: 24 audit entries account for 25 source records.
Ozone observation fixtures passed: 19 cases, 16 output records.
Temperature field fixtur

### 8.6 Publish this run

We save the validated bundle with its input versions, rules, code fingerprints and test results.

Save the notebook before running this section. The code fingerprint includes comments inside code cells, so documentation edits can change release metadata even when the processing rules stay the same.

In [36]:
# Publish the actual source Silver bundle after its saved-code and fixture checks pass.
source_silver_publication = None
if (
    source_silver_checks is None
    or source_silver_checks["run_id"] != silver_context.run_id
    or source_silver_checks["stage_sha256"]
    != silver_json_sha256(source_silver_stage)
    or source_silver_checks["status"] != "passed"
):
    raise RuntimeError(
        "Run 8.5 for this staged bundle before publishing."
    )

def saved_silver_code():
    """Return checksums of the saved code cells, source modules and Silver test file."""
    # Hash the saved code cells and companion files, rather than the notebook's displayed outputs.
    notebook = json.loads((project_dir / "notebooks/pipeline.ipynb").read_text(encoding="utf-8"))
    return {
        "notebook_code_sha256": {"pipeline.ipynb": notebook_code_sha256(notebook)},
        "source_files_sha256": {
            path.relative_to(project_dir).as_posix(): file_sha256(path)
            for path in [*sorted((project_dir / "src").glob("*.py")), fixture_file]
        },
    }


# Check that the test file has not changed since its fixtures ran.
saved_code = saved_silver_code()
if saved_code["source_files_sha256"]["tests/silver_checks.py"] != fixture_code_hash:
    raise ValueError("Silver test code changed. Restart the kernel and rerun the checks.")

source_silver_root = Path(
    silver_context.environment_root
).resolve()
supporting_paths = [
    project_dir / "config" / "spark.properties",
    environments_file,
]

# Record the code, source manifests, runtime and fixture evidence used for this release.
source_silver_provenance = {
    **capture_provenance(silver_context),
    "notebook_code_hash_scheme": "sha256-json-code-cells-v1",
    "saved_notebook_code_sha256": saved_code["notebook_code_sha256"],
    "supporting_files_sha256": saved_code["source_files_sha256"] | {
        path.relative_to(project_dir).as_posix(): file_sha256(path)
        for path in supporting_paths
    },
    "bronze_manifests_sha256": {
        item["lineage"]["source_manifest_path"]: file_sha256(
            silver_child_path(
                source_silver_root,
                item["lineage"]["source_manifest_path"],
                must_exist=True,
            )
        )
        for item in source_silver_validation["audits"].values()
    },
    "runtime": {
        "python": sys.version.split()[0],
        "pyspark": pyspark.__version__,
        "spark": spark.version,
        "master": spark.sparkContext.master,
        "session_timezone": spark.conf.get(
            "spark.sql.session.timeZone"
        ),
    },
    "fixture_checks": source_silver_checks,
}

# Move the validated bundle and write its publication manifest.
source_silver_publication = publish_source_silver(
    source_silver_stage,
    source_silver_validation,
    source_silver_contracts,
    silver_context,
    profile=alabama_profile,
    rule_versions=source_silver_rule_versions,
    provenance=source_silver_provenance,
)
# Keep the manifest checksum so the next loader can verify this exact release.
source_silver_publication["manifest_sha256"] = file_sha256(
    Path(source_silver_publication["manifest_path"])
)

# The staging paths are no longer valid after publication.
source_silver_readback = None

print("Publication outcome:", source_silver_publication["outcome"])
print(
    "Publication status:",
    source_silver_publication["manifest"]["status"],
)
print(
    "Published datasets:",
    len(source_silver_publication["manifest"]["datasets"]),
)
print("Manifest:", source_silver_publication["manifest_path"])
print(
    "Manifest SHA-256:",
    source_silver_publication["manifest_sha256"],
)
print("Fixture status:", source_silver_checks["status"])

Publication outcome: published
Publication status: published
Published datasets: 8
Manifest: /home/ubuntu/dat535-final-project-data/dev/published/silver/source/alabama/2025/6344bf8511c2501b98436457b77c39cf90cdc06a16e20cbd3262da54c9ad742a/manifest.json
Manifest SHA-256: edb12c417be5c243189a7f95d8b1cdafe8f202e65412dc022335dc18076b1741
Fixture status: passed


### 8.7 Reload source Silver

We use the manifest to reopen the eight tables and verify their files and counts. The hourly preparation below starts from this saved release.

In [37]:
# Reopen the published files so hourly preparation uses a saved, verified release.
source_silver_loaded_manifest = None
source_silver_loaded_tables = None
source_silver_loaded_counts = None
(
    source_silver_loaded_manifest,
    source_silver_loaded_tables,
    source_silver_loaded_counts,
) = load_source_silver(
    source_silver_publication["manifest_path"],
    silver_context,
    spark=spark,
    expected_manifest_sha256=(
        source_silver_publication["manifest_sha256"]
    ),
    profile=alabama_profile,
    rule_versions=source_silver_rule_versions,
)

# Display the counts returned by the verified loader.
spark.createDataFrame(
    sorted(source_silver_loaded_counts.items()),
    "dataset STRING, records LONG",
).show(truncate=False)

# Write a completion receipt only after all eight published tables have reloaded successfully.
source_silver_completion = {
    "status": "success",
    "stage": "source_silver",
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "run_id": silver_context.run_id,
    "environment": silver_context.environment,
    "manifest_path": (
        Path(source_silver_publication["manifest_path"])
        .relative_to(silver_context.environment_root)
        .as_posix()
    ),
    "manifest_sha256": source_silver_publication["manifest_sha256"],
    "publication_outcome": source_silver_publication["outcome"],
    "datasets": source_silver_loaded_counts,
    "fixture_status": source_silver_checks["status"],
    "fixture_results": source_silver_checks["results"],
}
# Save the receipt with this run's reports for the runner and later inspection.
source_silver_completion_path = (
    silver_context.reports_dir / "source_silver_completion.json"
)
write_json(
    source_silver_completion_path, source_silver_completion
)

print(
    "Source Silver loaded and verified:",
    len(source_silver_loaded_tables),
    "datasets.",
)
print("Source Silver status:", source_silver_completion["status"])
print("Fixture status:", source_silver_checks["status"])
print(
    "Profile status:",
    source_silver_loaded_manifest["profile"]["status"],
)
print("Completion report:", source_silver_completion_path)

+-------------------+-------+
|dataset            |records|
+-------------------+-------+
|epa_monitors       |1      |
|epa_sites          |1      |
|noaa_stations      |1      |
|ozone_audit        |5166   |
|ozone_observations |5071   |
|reference_audit    |3      |
|temperature_audit  |11851  |
|temperature_reports|11686  |
+-------------------+-------+

Source Silver loaded and verified: 8 datasets.
Source Silver status: success
Fixture status: passed
Profile status: provisional
Completion report: /home/ubuntu/dat535-final-project-data/dev/runs/bc50a11d476d4e0083d429448023dcef/reports/source_silver_completion.json


## 9. Silver: match ozone and temperature by hour

An EPA timestamp marks the start of an ozone hour. We select the latest eligible NOAA report in that same UTC hour. A left join keeps the ozone reading when no temperature is available.

`prepare_hourly_silver` in `src/silver_hourly.py` also builds a calendar for the whole year. This lets us distinguish an hour with an excluded reading from an hour with no reading at all.

The helper returns six tables: the yearly monitor-station mapping, one temperature report per hour, ozone with its matched temperature, a full hourly calendar, monthly coverage and gap intervals. Here `hourly_tables` still refers to the reloaded source tables; the new hourly outputs are in `prepared_silver["tables"]`.

In [38]:
# Start from the source Silver files reloaded in 8.7, rather than the earlier in-memory outputs.
hourly_manifest = source_silver_loaded_manifest
hourly_tables = source_silver_loaded_tables
hourly_profile = hourly_manifest["profile"]
# prepare_hourly_silver returns a dictionary containing six tables and their checks.
# The transformations themselves are in src/silver_hourly.py.
prepared_silver = prepare_hourly_silver(
    hourly_tables, hourly_manifest,
    environment=silver_context.environment,
)

# Display coverage and checks. These prepared tables have not been published yet.
print("Preparation status:", prepared_silver["status"])
print("In-memory checks:", prepared_silver["checks"]["status"])
print("Region and year:", prepared_silver["profile"]["region"], prepared_silver["profile"]["year"])
print("Eligible ozone hours:", prepared_silver["checks"]["coverage"]["eligible_ozone_hours"])
print("Eligible temperature hours:", prepared_silver["checks"]["coverage"]["eligible_temperature_hours"])
print("Shared hours:", prepared_silver["checks"]["coverage"]["shared_hours"])
print("Calendar hours:", prepared_silver["checks"]["coverage"]["calendar_hours"])
print("Months:", prepared_silver["checks"]["monthly"]["months"])
print("Mapping approval:", prepared_silver["mapping_evidence"]["approval_status"])

Preparation status: prepared_unpublished
In-memory checks: passed
Region and year: Alabama 2025
Eligible ozone hours: 5071
Eligible temperature hours: 8514
Shared hours: 4836
Calendar hours: 8760
Months: 12
Mapping approval: pending


### 9.1 Inspect the hourly match and coverage

The first example shows which weather report was chosen in an hour with several candidates. The next two show an ozone hour with missing temperature and a month with low ozone coverage.

In [39]:
# Inspect three examples that explain the hourly selection and coverage.
tables = prepared_silver["tables"]
# Find the first hour with more than one eligible temperature report.
choice = (
    tables["temperature_hourly"].where(F.col("qualifying_report_count") > 1)
    .orderBy("hour_start_utc")
    .select("station_id", "hour_start_utc", "report_timestamp_utc")
    .limit(1)
)
# Show its candidate reports and mark the report selected for that hour.
candidates = (
    hourly_tables["temperature_reports"]
    .withColumn("hour_start_utc", F.date_trunc("hour", "timestamp_utc"))
    .join(choice, ["station_id", "hour_start_utc"])
    .select(
        "timestamp_utc", "temperature_c",
        (F.col("timestamp_utc") == F.col("report_timestamp_utc")).alias("selected"),
    )
)
print("Eligible reports in the first multi-report hour (empty if none):")
candidates.orderBy("timestamp_utc").show(truncate=False)

# Show one ozone hour retained with missing weather.
tables["ozone_enriched"].where(~F.col("temperature_available")).select(
    "monitor_id", "hour_start_utc", "ozone_ppm", "temperature_c", "integration_flags"
).orderBy("hour_start_utc").show(1, truncate=False)
# Show the month with the lowest eligible ozone coverage, including months with none.
tables["monthly_coverage"].select(
    "month_start_utc", "calendar_hours", "eligible_ozone_hours", "shared_hours", "ozone_calendar_pct"
).orderBy("ozone_calendar_pct", "month_start_utc").show(1, truncate=False)

Eligible reports in the first multi-report hour (empty if none):
+-------------------+-------------+--------+
|timestamp_utc      |temperature_c|selected|
+-------------------+-------------+--------+
|2025-01-05 15:15:00|9.4          |false   |
|2025-01-05 15:53:00|8.9          |true    |
+-------------------+-------------+--------+

+-------------------+-------------------+---------+-------------+-------------------------+
|monitor_id         |hour_start_utc     |ozone_ppm|temperature_c|integration_flags        |
+-------------------+-------------------+---------+-------------+-------------------------+
|01-073-1005-44201-1|2025-06-01 14:00:00|0.04     |NULL         |[no_eligible_temperature]|
+-------------------+-------------------+---------+-------------+-------------------------+
only showing top 1 row

+-------------------+--------------+--------------------+------------+------------------+
|month_start_utc    |calendar_hours|eligible_ozone_hours|shared_hours|ozone_calendar_pct|


### 9.2 Test hourly integration

The test cases check the selected report, an ozone hour with missing weather, and a leap-year calendar.

In [40]:
# Run small cases for hourly selection, missing weather and calendar coverage.
hourly_fixture_result = silver_checks.verify_hourly_integration(
    spark=spark,
    profile=hourly_profile,
    references=hourly_tables,
    build_yearly_mapping=build_yearly_mapping,
    select_hourly_temperature=select_hourly_temperature,
    enrich_hourly_ozone=enrich_hourly_ozone,
    prepare_hourly_silver=prepare_hourly_silver,
)
# Keep the fixture result and code hash as evidence for the hourly publication.
hourly_fixture_checks = {
    "status": "passed",
    "fixture_file_sha256": fixture_code_hash,
    "results": hourly_fixture_result,
}
print("Hourly integration fixtures passed:", hourly_fixture_result)

Hourly integration fixtures passed: {'checks': 29, 'calendar_hours': 8784, 'months': 12}


### 9.3 How well does the Alabama pairing cover 2025?

McAdory had eligible ozone readings in **5,071 of 8,760 calendar hours (57.89%)**. Of those 5,071 hours, **4,836 (95.37%)** had eligible temperature. The other 235 ozone readings stay in the dataset with a missing temperature.

We recorded the McAdory–Alabaster pairing as `accepted_with_limitations` for these available observations. The decision is tied to the profile, mapping, rules and source release. The weather station is about 26.7 km away, so the results describe this pairing. The historical operating schedule is unknown, and we cannot tell from a gap alone why readings are absent.

This decision applies to the reviewed Alabama pair only. We save it separately from the profile JSON and attach its checksum to the mapping. It does not approve Georgia or turn missing observations into measured hours.

In [41]:
# Record the reviewed Alabama pairing decision for this exact preparation.
alabama_silver_acceptance = None

# Clear any previous approval before checking the current mapping.
decision_mapping = prepared_silver["mapping_evidence"]
decision_mapping["approval_status"] = "pending"
decision_mapping.pop("approval_decision_sha256", None)

# Restrict this decision to the reviewed McAdory-Alabaster pair in 2025.
if (
    hourly_profile["region"] != "Alabama"
    or hourly_profile["year"] != 2025
    or prepared_silver["mapping_evidence"]["monitor_id"]
    != "01-073-1005-44201-1"
    or prepared_silver["mapping_evidence"]["station_id"]
    != "USW00053864"
):
    raise ValueError(
        "This acceptance decision applies to the reviewed Alabama pairing."
    )

# Require preparation and fixture checks to pass before recording acceptance.
if prepared_silver["checks"]["status"] != "passed":
    raise ValueError("In-memory preparation checks must pass.")

if hourly_fixture_checks["status"] != "passed":
    raise ValueError(
        "Run the hourly integration fixtures before recording acceptance."
    )

# Unresolved mapping flags need review; the code does not approve them automatically.
decision_mapping_flags = sorted(set(decision_mapping["review_flags"]))
if decision_mapping_flags:
    raise ValueError(
        "Mapping metadata requires review before acceptance: "
        + ", ".join(decision_mapping_flags)
    )

# Save the observed coverage and limitations with the profile, source and rules that were reviewed.
decision_coverage = prepared_silver["checks"]["coverage"]
alabama_silver_acceptance = {
    "decision_version": "available-observations-v1",
    "status": "accepted_with_limitations",
    "region": hourly_profile["region"],
    "year": hourly_profile["year"],
    "profile_sha256": silver_json_sha256(prepared_silver["profile"]),
    "source_silver_publication_id": (
        prepared_silver["source_silver_publication_id"]
    ),
    "mapping_id": prepared_silver["mapping_evidence"]["mapping_id"],
    "mapping_review_flags": decision_mapping_flags,
    "rules_sha256": silver_json_sha256(prepared_silver["rules"]),
    "coverage": {
        name: decision_coverage[name]
        for name in (
            "calendar_hours",
            "eligible_ozone_hours",
            "eligible_temperature_hours",
            "shared_hours",
        )
    },
    "critical_failures": [
        "Incompatible or unverifiable source publication.",
        "Missing, ambiguous or invalid required mapping.",
        "Mapping metadata flags requiring review.",
        "Unusable or repeated output keys, or row multiplication.",
        "Unreconciled source audits, calendar counts or gap intervals.",
        "Failed integration fixtures or publication read-back checks.",
    ],
    "reportable_limitations": [
        "Missing and excluded observations remain visible across all twelve months.",
        "Missing temperature remains null; eligible ozone is retained.",
        "Historical operating schedules are unknown.",
        (
            "Monitor-station distance: "
            f"{prepared_silver['mapping_evidence']['distance_km']:.2f} km."
        ),
        "Results describe the selected monitor-station pair and available observations.",
        "Observed gaps do not establish seasonal operation or instrument downtime.",
    ],
    "publication_condition": (
        "All publication and read-back checks must pass."
    ),
}

# Write the decision separately from the provisional profile JSON.
alabama_acceptance_path = (
    silver_context.reports_dir / "alabama_silver_acceptance.json"
)
write_json(alabama_acceptance_path, alabama_silver_acceptance)

# Attach this decision's checksum to the mapping. A changed decision must not
# be mistaken for the one used when the hourly files were prepared.
prepared_silver["mapping_evidence"] = {
    **prepared_silver["mapping_evidence"],
    "approval_status": "accepted_with_limitations",
    "approval_decision_sha256": silver_json_sha256(
        alabama_silver_acceptance
    ),
}

print("Alabama decision:", alabama_silver_acceptance["status"])
print("Eligible ozone hours:", decision_coverage["eligible_ozone_hours"])
print("Shared hours:", decision_coverage["shared_hours"])
print("Publication remains pending read-back validation.")
print("Decision report:", alabama_acceptance_path)

Alabama decision: accepted_with_limitations
Eligible ozone hours: 5071
Shared hours: 4836
Publication remains pending read-back validation.
Decision report: /home/ubuntu/dat535-final-project-data/dev/runs/bc50a11d476d4e0083d429448023dcef/reports/alabama_silver_acceptance.json


### 9.4 Publish hourly Silver

The bundle contains the yearly mapping, hourly temperatures, ozone with temperature, hourly coverage, monthly coverage and gap intervals.

#### 9.4a Stage the six tables

We check that the acceptance decision matches this preparation and its source Silver release before writing the files.

In [42]:
# Stage the six hourly tables only after their source, code and acceptance evidence agree.
hourly_publication = None
hourly_loaded_tables = None
hourly_validation = None
hourly_stage = None

# Require the prepared tables to belong to the selected source Silver publication.
if (
    prepared_silver["status"] != "prepared_unpublished"
    or prepared_silver["profile"] != hourly_manifest["profile"]
    or prepared_silver["source_silver_publication_id"]
    != hourly_manifest["publication_id"]
    or set(prepared_silver["tables"]) != set(hourly_output_keys)
):
    raise ValueError("Prepare hourly Silver from the selected source publication.")

# Check that saved code and fixture results still describe this preparation.
hourly_saved_code = saved_silver_code()
if (
    hourly_saved_code != saved_code
    or hourly_fixture_checks["status"] != "passed"
    or hourly_saved_code["source_files_sha256"]["tests/silver_checks.py"]
    != hourly_fixture_checks["fixture_file_sha256"]
):
    raise ValueError("Run the hourly fixtures against the saved companion code.")

# Save an independent copy of the settings, acceptance and parent-source details.
# These are the evidence used to approve this particular hourly bundle.
hourly_evidence = json.loads(json.dumps({
    "profile": prepared_silver["profile"],
    "rules": prepared_silver["rules"],
    "mapping": prepared_silver["mapping_evidence"],
    "acceptance": alabama_silver_acceptance,
    "checks": prepared_silver["checks"],
    "fixtures": hourly_fixture_checks,
    "source": {
        "publication_id": hourly_manifest["publication_id"],
        "manifest_path": Path(source_silver_publication["manifest_path"])
        .relative_to(silver_context.environment_root).as_posix(),
        "manifest_sha256": source_silver_publication["manifest_sha256"],
        "rule_versions": hourly_manifest["rule_versions"],
    },
    "provenance": {
        **capture_provenance(silver_context),
        "saved_notebook_code_sha256": hourly_saved_code["notebook_code_sha256"],
        "source_files_sha256": hourly_saved_code["source_files_sha256"],
        "supporting_files_sha256": {
            path.relative_to(project_dir).as_posix(): file_sha256(path)
            for path in supporting_paths
        },
        "python": sys.version.split()[0],
        "spark": spark.version,
        "master": spark.sparkContext.master,
        "timezone": spark.conf.get("spark.sql.session.timeZone"),
    },
}, allow_nan=False))
# Check that the approval and preparation evidence refer to the same mapping and source.
check_hourly_evidence(hourly_evidence)

# Reopen the parent source release and confirm it is still the one we prepared from.
verified_parent, _, _ = load_source_silver(
    source_silver_publication["manifest_path"], silver_context,
    spark=spark,
    expected_manifest_sha256=hourly_evidence["source"]["manifest_sha256"],
    profile=hourly_evidence["profile"],
    rule_versions=hourly_evidence["source"]["rule_versions"],
)
if verified_parent != hourly_manifest:
    raise ValueError("The selected source publication changed.")

# Define each hourly table's schema, key and expected count from the preparation evidence.
hourly_row_counts = hourly_expected_rows(hourly_evidence)
hourly_contracts = {
    name: {
        "schema_version": f"{name}-v1",
        "columns": dict(table.dtypes),
        "column_order": table.columns,
        "key": hourly_output_keys[name],
        "expected_rows": hourly_row_counts[name],
    }
    for name, table in prepared_silver["tables"].items()
}

# Reuse the Parquet staging helper to write the six hourly tables.
hourly_stage = stage_source_silver(
    prepared_silver["tables"], hourly_contracts, silver_context,
)
print("Hourly status: staged, unpublished")

Staged yearly_mapping: 1 rows, 1 Parquet file(s).
Staged temperature_hourly: 8514 rows, 1 Parquet file(s).
Staged ozone_enriched: 5071 rows, 1 Parquet file(s).
Staged hourly_coverage: 8760 rows, 1 Parquet file(s).
Staged monthly_coverage: 12 rows, 1 Parquet file(s).
Staged hourly_gaps: 143 rows, 1 Parquet file(s).
Hourly status: staged, unpublished


#### 9.4b Compare the saved tables with the preparation

We read the Parquet files back and compare every column and row, including missing values, flags and source details. Matching row counts alone could miss a different weather report being selected.

In [43]:
# Read the saved hourly tables back and compare them with the prepared tables.
hourly_validation = None
# Require the stage, contracts and acceptance evidence to describe the same preparation.
check_hourly_evidence(hourly_evidence)
if (
    hourly_stage["run_id"] != silver_context.run_id
    or hourly_stage["environment"] != silver_context.environment
    or hourly_stage["contract_sha256"] != silver_json_sha256(hourly_contracts)
    or {name: item["expected_rows"] for name, item in hourly_contracts.items()}
    != hourly_expected_rows(hourly_evidence)
):
    raise ValueError("Stage, contracts and preparation evidence do not match.")

# The loader checks files and keys, then compares all columns and rows in both directions.
hourly_readback, hourly_dataset_checks = read_hourly_files(
    silver_child_path(
        silver_context.environment_root,
        hourly_stage["bundle_path"],
        must_exist=True,
    ),
    hourly_stage["datasets"],
    hourly_contracts,
    spark=spark,
    expected=prepared_silver["tables"],
)

# Record which staged files, contracts and evidence passed these comparisons.
# The publisher rejects this report if any of them have changed.
hourly_validation = {
    "status": "validated",
    "validated_at_utc": datetime.now(timezone.utc).isoformat(),
    "run_id": silver_context.run_id,
    "stage_sha256": silver_json_sha256(hourly_stage),
    "contract_sha256": silver_json_sha256(hourly_contracts),
    "evidence_sha256": silver_json_sha256(hourly_evidence),
    "datasets": hourly_dataset_checks,
}
# Save validation evidence before attempting publication.
write_json(
    silver_context.reports_dir / "hourly_silver_validation.json",
    hourly_validation,
)
print("Hourly status: validated, unpublished")
print("Complete table comparisons passed:", len(hourly_dataset_checks))

Verified hourly_coverage: 8760 rows.
Verified hourly_gaps: 143 rows.
Verified monthly_coverage: 12 rows.
Verified ozone_enriched: 5071 rows.
Verified temperature_hourly: 8514 rows.
Verified yearly_mapping: 1 rows.
Hourly status: validated, unpublished
Complete table comparisons passed: 6


#### 9.4c Publish the validated bundle

The manifest is written after the file checks pass. Repeating the same publication attempt should return the same manifest. The cell below also checks that a failed validation blocks publication.

In [44]:
# Define publication for a validated hourly bundle, then exercise the guard and retry below.
def publish_hourly_silver(stage, validation, contracts, evidence, context):
    """Publish the validated hourly tables and write the manifest after the files are in place."""
    # Copy the supplied metadata and reject values that cannot be recorded as standard JSON.
    stage, validation, contracts, evidence = json.loads(json.dumps(
        [stage, validation, contracts, evidence], allow_nan=False,
    ))
    # Check the reviewed mapping and source parent before any files are moved.
    check_hourly_evidence(evidence)
    stage_hash = silver_json_sha256(stage)
    contract_hash = silver_json_sha256(contracts)
    evidence_hash = silver_json_sha256(evidence)
    root = Path(context.environment_root).resolve()

    # Require validation to match this run, staged bundle, contracts and evidence.
    if (
        stage["status"] != "staged"
        or stage["run_id"] != context.run_id
        or stage["environment"] != context.environment
        or Path(stage["environment_root"]).resolve() != root
        or stage["contract_sha256"] != contract_hash
        or validation["status"] != "validated"
        or validation["run_id"] != context.run_id
        or validation["stage_sha256"] != stage_hash
        or validation["contract_sha256"] != contract_hash
        or validation["evidence_sha256"] != evidence_hash
        or set(stage["datasets"]) != set(hourly_output_keys)
        or set(contracts) != set(hourly_output_keys)
        or set(validation["datasets"]) != set(hourly_output_keys)
        or {name: item["expected_rows"] for name, item in contracts.items()}
        != hourly_expected_rows(evidence)
    ):
        raise ValueError(
            "Publication requires matching stage, validation and evidence."
        )

    # Matching counts are not enough: every table must also match every prepared row and column.
    for name, info in stage["datasets"].items():
        check = validation["datasets"][name]
        if (
            check["status"] != "passed"
            or check["complete_values_match"] is not True
            or check["records"] != info["row_count"]
            or check["records"] != contracts[name]["expected_rows"]
            or check["parquet_files"] != len(info["files"])
        ):
            raise ValueError(
                f"{name}: complete read-back validation is required."
            )

    # Validate the region and year used in the release path.
    region = evidence["profile"]["region"].strip().lower().replace(" ", "_")
    year = evidence["profile"]["year"]
    if (
        not region
        or any(c not in "abcdefghijklmnopqrstuvwxyz0123456789_-" for c in region)
        or type(year) is not int or not 1 <= year <= 9999
    ):
        raise ValueError("Invalid region or year.")

    # Keep the move within the selected environment and this run's staging directory.
    staged = silver_child_path(root, stage["bundle_path"])
    staging_root = Path(context.staging_root).resolve()
    if (
        not staging_root.is_relative_to(root / "staging")
        or staged == staging_root
        or not staged.is_relative_to(staging_root)
    ):
        raise ValueError(
            "The bundle must be inside this run's staging directory."
        )

    # Identify the publication from both the staged bundle and its supporting evidence.
    publication_id = silver_json_sha256({
        "stage_sha256": stage_hash,
        "evidence_sha256": evidence_hash,
    })
    suffix = f"silver/hourly/{region}/{year}/{publication_id}"
    final_bundle = silver_child_path(root, f"data/{suffix}")
    manifest_path = silver_child_path(root, f"published/{suffix}/manifest.json")
    publication_root = silver_child_path(root, "published/silver/hourly")

    # Record the six datasets, their contracts, validation and parent-source evidence.
    identity = {
        "manifest_version": 1,
        "kind": "hourly_silver",
        "status": "published",
        "publication_id": publication_id,
        "run_id": context.run_id,
        "environment": context.environment,
        "bundle_path": final_bundle.relative_to(root).as_posix(),
        "stage_sha256": stage_hash,
        "contract_sha256": contract_hash,
        "evidence_sha256": evidence_hash,
        "contracts": contracts,
        "datasets": stage["datasets"],
        "evidence": evidence,
        "validation": validation,
        "storage": {
            key: stage[key]
            for key in (
                "format", "compression", "write_partitions",
                "partition_columns",
            )
        },
    }

    # Lock hourly publication so two runs cannot move files and write manifests at the same time.
    publication_root.mkdir(parents=True, exist_ok=True)
    lock = publication_root / ".publish.lock"
    try:
        lock.mkdir()
    except FileExistsError as error:
        raise RuntimeError(
            f"Publication lock already exists: {lock}"
        ) from error

    try:
        # Reuse an identical attempt only after its saved metadata and files have been checked.
        if manifest_path.exists():
            existing = json.loads(manifest_path.read_text(encoding="utf-8"))
            if {key: existing.get(key) for key in identity} != identity:
                raise ValueError(
                    "The existing publication has different metadata."
                )
            verify_silver_file_inventory(
                final_bundle, existing["datasets"]
            )
            outcome, manifest = "reused", existing
        else:
            # An existing data directory without a manifest is incomplete and needs inspection.
            if final_bundle.exists() or final_bundle.is_symlink():
                raise FileExistsError(
                    f"An incomplete publication exists at {final_bundle}. "
                    "Inspect it, then stage a new attempt."
                )
            # Verify staged files and require a same-filesystem move to the final location.
            verify_silver_file_inventory(staged, stage["datasets"])
            final_bundle.parent.mkdir(parents=True, exist_ok=True)
            if staged.stat().st_dev != final_bundle.parent.stat().st_dev:
                raise ValueError(
                    "Staging and publication require the same filesystem."
                )

            manifest = {
                **identity,
                "published_at_utc": datetime.now(timezone.utc).isoformat(),
            }
            # Move the data first. The manifest makes the complete release available to readers.
            staged.rename(final_bundle)
            write_json(manifest_path, manifest)
            outcome = "published"

        return {
            "outcome": outcome,
            "manifest_path": str(manifest_path),
            "manifest_sha256": file_sha256(manifest_path),
            "manifest": manifest,
        }
    # Always release the publication lock, including when an error is raised.
    finally:
        lock.rmdir()


# Check the saved code again before publishing the actual hourly bundle.
hourly_publication = None
if (
    saved_silver_code() != hourly_saved_code
):
    raise ValueError(
        "Saved code changed after staging. Rerun the hourly staging and validation cells."
    )

# Failed validation must be rejected before publication.
try:
    publish_hourly_silver(
        hourly_stage,
        {**hourly_validation, "status": "failed"},
        hourly_contracts,
        hourly_evidence,
        silver_context,
    )
except ValueError:
    print("Failed-validation guard: passed")
else:
    raise AssertionError("Failed validation was incorrectly accepted.")

# Publish the validated bundle with its source and acceptance evidence.
hourly_publication = publish_hourly_silver(
    hourly_stage, hourly_validation,
    hourly_contracts, hourly_evidence, silver_context,
)
# Call the publisher again with the same stage and evidence to test safe retries.
# This checks one identical attempt, not whether a full new run reuses its publication.
hourly_retry = publish_hourly_silver(
    hourly_stage, hourly_validation,
    hourly_contracts, hourly_evidence, silver_context,
)
if (
    hourly_retry["outcome"] != "reused"
    or hourly_retry["manifest_sha256"]
    != hourly_publication["manifest_sha256"]
):
    raise AssertionError(
        "The identical publication attempt was not reused."
    )

# Drop the stale staging reference; the next cell reloads from the published paths.
hourly_readback = None
print("Publication outcome:", hourly_publication["outcome"])
print("Identical-attempt reuse: passed")
print("Manifest:", hourly_publication["manifest_path"])
print("Manifest SHA-256:", hourly_publication["manifest_sha256"])

Failed-validation guard: passed
Publication outcome: published
Identical-attempt reuse: passed
Manifest: /home/ubuntu/dat535-final-project-data/dev/published/silver/hourly/alabama/2025/c22c1b3b6f61e6896a09e5845469903776890bb2a63b3af96f8c519c73ad63b9/manifest.json
Manifest SHA-256: ad34b6a43e297b609ae1111763fa5379da5f542cb0f11fa3e8cbbcf26a9d0543


#### 9.4d Reload hourly Silver

The loader verifies the manifest, its parent source release and the six tables. The completion report gives the manifest path and checksum Gold can use to open this exact dataset.

Prepare mode ends at this cell. Analyse uses the saved receipt to reopen the same hourly release in a new process. Full continues to Gold with these already verified tables.

In [45]:
# Reopen and verify the published hourly release before handing it to Gold.
hourly_loaded_manifest = None
hourly_loaded_tables = None
hourly_silver_completion = None

(
    hourly_loaded_manifest,
    hourly_loaded_tables,
    hourly_load_checks,
) = load_hourly_silver(
    hourly_publication["manifest_path"],
    silver_context,
    spark=spark,
    expected_manifest_sha256=hourly_publication["manifest_sha256"],
    profile=hourly_profile,
)

# Record the release ID, manifest checksum and source parent in the hourly receipt.
hourly_silver_completion = {
    "status": "success",
    "readiness": "prepared",
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "run_id": silver_context.run_id,
    "environment": silver_context.environment,
    "publication_id": hourly_loaded_manifest["publication_id"],
    "manifest_path": Path(hourly_publication["manifest_path"])
    .relative_to(silver_context.environment_root).as_posix(),
    "manifest_sha256": hourly_publication["manifest_sha256"],
    "source_silver_publication_id": (
        hourly_loaded_manifest["evidence"]["source"]["publication_id"]
    ),
    "datasets": {
        name: check["records"]
        for name, check in hourly_load_checks.items()
    },
}
# Save this receipt for the runner and for a later Analyse invocation.
hourly_completion_path = (
    silver_context.reports_dir / "hourly_silver_completion.json"
)
write_json(hourly_completion_path, hourly_silver_completion)

# Display the verified table counts and the receipt location.
spark.createDataFrame(
    sorted(hourly_silver_completion["datasets"].items()),
    "dataset STRING, records LONG",
).show(truncate=False)

print("Hourly Silver status: prepared")
print("Published datasets reloaded:", len(hourly_loaded_tables))
print("Completion report:", hourly_completion_path)
print("Gold input manifest:", hourly_publication["manifest_path"])
print(
    "Gold input manifest SHA-256:",
    hourly_publication["manifest_sha256"],
)

Verified hourly_coverage: 8760 rows.
Verified hourly_gaps: 143 rows.
Verified monthly_coverage: 12 rows.
Verified ozone_enriched: 5071 rows.
Verified temperature_hourly: 8514 rows.
Verified yearly_mapping: 1 rows.
+------------------+-------+
|dataset           |records|
+------------------+-------+
|hourly_coverage   |8760   |
|hourly_gaps       |143    |
|monthly_coverage  |12     |
|ozone_enriched    |5071   |
|temperature_hourly|8514   |
|yearly_mapping    |1      |
+------------------+-------+

Hourly Silver status: prepared
Published datasets reloaded: 6
Completion report: /home/ubuntu/dat535-final-project-data/dev/runs/bc50a11d476d4e0083d429448023dcef/reports/hourly_silver_completion.json
Gold input manifest: /home/ubuntu/dat535-final-project-data/dev/published/silver/hourly/alabama/2025/c22c1b3b6f61e6896a09e5845469903776890bb2a63b3af96f8c519c73ad63b9/manifest.json
Gold input manifest SHA-256: ad34b6a43e297b609ae1111763fa5379da5f542cb0f11fa3e8cbbcf26a9d0543


In [46]:
# Reload local Gold modules when working interactively after saving helper edits.
# This reloads module definitions; importing gold_checks does not run its tests.
# It does not recompute existing DataFrames. Restart and rerun for a clean verification.
# Prepare stops before this cell, and the Analyse runner skips this development aid.
import importlib
import gold
import gold_checks
import gold_io

importlib.reload(gold)
importlib.reload(gold_checks)
importlib.reload(gold_io)

<module 'gold_io' from '/home/ubuntu/dat535-final-project-work/group02-dat535/final_project/src/gold_io.py'>

## 10. Gold: sustained ozone periods in 2025

### 10.1 Rule and input

Our primary rule, `gold-period-v1`, is ozone **at or above 0.045 ppm for at least three consecutive UTC hours**. The cutoff was the 90th percentile of the eligible McAdory readings used for calibration. We saved it in `config/gold_rules.json`; this cell does not recalculate P90.

A missing, excluded or below-cutoff ozone hour breaks the sequence. Missing temperature does not break it. In Full mode, we use the hourly Silver tables verified in 9.4d. In Analyse mode, we reopen the release named in a saved receipt and can select another saved rule.

The calibration source records where the cutoff came from. The printed Silver parent identifies the data being analysed now. The calculations are in `src/gold.py`, and publication and reloading are in `src/gold_io.py`.

In [47]:
# Select the saved Gold rule and the verified hourly Silver input.
from gold import load_gold_rules

# Use the primary rule unless Analyse supplied an alternative saved rule file.
rule_path = Path(os.environ.get("PIPELINE_GOLD_RULES", project_dir / "config/gold_rules.json"))
gold_rules = load_gold_rules(rule_path)

# Analyse supplies a receipt; Full uses the hourly tables just reloaded in section 9.
receipt_path = os.environ.get("PIPELINE_HOURLY_RECEIPT")

# For Analyse, accept only a successful hourly receipt from this environment's runs directory.
if receipt_path:
    path = Path(receipt_path).resolve(strict=True)
    if path.name != "hourly_silver_completion.json" or not path.is_relative_to(
        silver_context.environment_root / "runs"
    ):
        raise ValueError("Select an hourly Silver receipt from this environment.")

    receipt = json.loads(path.read_text(encoding="utf-8"))
    if receipt["status"] != "success" or receipt["environment"] != silver_context.environment:
        raise ValueError("The selected hourly Silver run was not successful in this environment.")

    # Verify and open the saved hourly release, including its files and parent source.
    # This entry point still requires the Alabama profile.
    gold_parent_path = receipt["manifest_path"]
    gold_parent_sha = receipt["manifest_sha256"]
    gold_parent, gold_inputs, _ = load_hourly_silver(
        silver_child_path(silver_context.environment_root, gold_parent_path, must_exist=True),
        silver_context,
        spark=spark,
        expected_manifest_sha256=gold_parent_sha,
        profile=alabama_profile,
    )
    # Require the loaded release ID to match the receipt as well as its manifest checksum.
    if gold_parent["publication_id"] != receipt["publication_id"]:
        raise ValueError("The hourly Silver publication differs from the receipt.")
else:
    # Full continues with the verified hourly tables and receipt from section 9.
    gold_parent = hourly_loaded_manifest
    gold_inputs = hourly_loaded_tables
    gold_parent_path = hourly_silver_completion["manifest_path"]
    gold_parent_sha = hourly_silver_completion["manifest_sha256"]

# Display the actual rule and Silver parent so an alternative-rule run is easy to identify.
print("Rule:", gold_rules["rule_version"], gold_rules["comparison"],
      gold_rules["threshold_ppm"], "ppm for at least", gold_rules["minimum_hours"], "hours")
print("Hourly Silver:", gold_parent["environment"], gold_parent["publication_id"])

Rule: gold-period-v1 >= 0.045 ppm for at least 3 hours
Hourly Silver: dev c22c1b3b6f61e6896a09e5845469903776890bb2a63b3af96f8c519c73ad63b9


### 10.2a Find consecutive elevated hours

We order the elevated ozone readings by monitor and UTC hour. A gap of more than one hour starts a new run. We keep every hour in a run that meets the duration rule, including runs that cross midnight or the end of a month.

The detector compares timestamps, not just neighbouring row positions. For example, elevated readings at 10:00, 11:00 and 13:00 form a two-hour run and a one-hour run because 12:00 is missing from the elevated readings. Missing temperature does not break an ozone run.

In [48]:
# Label consecutive elevated ozone hours using the detector in src/gold.py.
from gold import elevated_runs
from pyspark.sql import functions as F

# Cache the labelled hours because the catalogue, coverage and sensitivity cells reuse them.
elevated = elevated_runs(gold_inputs["ozone_enriched"], gold_rules).cache()
# Keep all hours from runs that meet the minimum duration.
# A five-hour run contributes five member rows, not three overlapping three-hour periods.
members = elevated.where(F.col("run_hours") >= gold_rules["minimum_hours"])
print("Elevated hours:", elevated.count())
print("Hours in qualifying periods:", members.count())

Elevated hours: 536
Hours in qualifying periods: 483


### 10.2b What happened during one period?

Each row in the catalogue describes one period. We report its duration, peak and mean ozone, and the temperatures available during those hours. The end time is the start of the first hour outside the period, whether or not that hour has an observation.

We also check the hour on either side. This shows whether we observed a lower ozone reading, had missing or excluded data, or reached the edge of the year.

For example, five member hours starting at 18:00, 19:00, 20:00, 21:00 and 22:00 form the interval from 18:00 up to, but not including, 23:00. Temperature averages use the available temperature values, and `temperature_hours` shows how many there were.

In [49]:
# Summarise the qualifying hours into one catalogue row per period.
from pyspark.sql import functions as F
from gold import build_period_catalogue

# Build period summaries and inspect the neighbouring hours to explain each boundary.
catalogue = build_period_catalogue(
    members, gold_inputs["ozone_enriched"], gold_inputs["hourly_coverage"], gold_rules
)
# Display the period count and total member hours for inspection.
catalogue.agg(
    F.count("*").alias("periods"),
    F.sum("duration_hours").alias("member_hours"),
).show()

# Display the first period so its summary can be compared with its member hours.
example = catalogue.orderBy("period_start_utc", "monitor_id").limit(1)
example.select(
    "period_start_utc", "period_end_utc", "duration_hours",
    "peak_ozone_ppm", "mean_ozone_ppm", "temperature_hours", "mean_temperature_c",
    "start_boundary", "end_boundary",
).show(truncate=False)

# Show up to six hours from that example; longer periods have more members than shown here.
period_key = ["region", "year", "monitor_id", "period_start_utc"]
(
    members.join(example.select(*period_key), period_key)
    .select("hour_start_utc", "ozone_ppm", "temperature_c")
    .orderBy("hour_start_utc")
    .show(6, truncate=False)
)

+-------+------------+
|periods|member_hours|
+-------+------------+
|     74|         483|
+-------+------------+

+-------------------+-------------------+--------------+--------------+--------------------+-----------------+------------------+---------------------+---------------------+
|period_start_utc   |period_end_utc     |duration_hours|peak_ozone_ppm|mean_ozone_ppm      |temperature_hours|mean_temperature_c|start_boundary       |end_boundary         |
+-------------------+-------------------+--------------+--------------+--------------------+-----------------+------------------+---------------------+---------------------+
|2025-04-08 18:00:00|2025-04-08 23:00:00|5             |0.047         |0.045799999999999993|5                |17.56             |observed_non_elevated|observed_non_elevated|
+-------------------+-------------------+--------------+--------------+--------------------+-----------------+------------------+---------------------+---------------------+

+------------

### 10.3 Monthly observation coverage

We keep all twelve months from Silver so we can see how much ozone data was available, including months with no qualifying periods. We count each period in the month when it starts. If it crosses into another month, each hour counts in the month when it occurred.

In [50]:
# Add Gold results to the existing twelve-month Silver coverage table.
from gold import build_monthly_coverage, summarise_year

# Count hours in their actual month and each period in the month when it starts.
months = build_monthly_coverage(
    gold_inputs["monthly_coverage"], elevated, members, catalogue
)
months.select(
    "month_start_utc", "calendar_hours", "eligible_ozone_hours", "shared_hours",
    "elevated_hours", "qualifying_period_hours", "periods_started",
).orderBy("month_start_utc").show(12, truncate=False)

# Pool the monthly counts for the year; percentages use pooled counts, not average percentages.
annual = summarise_year(months)
annual.select(
    "calendar_hours", "eligible_ozone_hours", "shared_hours",
    "ozone_hours_without_temperature", "temperature_match_pct",
    "elevated_hours", "qualifying_period_hours", "periods_started", "outcome",
).show(truncate=False)

+-------------------+--------------+--------------------+------------+--------------+-----------------------+---------------+
|month_start_utc    |calendar_hours|eligible_ozone_hours|shared_hours|elevated_hours|qualifying_period_hours|periods_started|
+-------------------+--------------+--------------------+------------+--------------+-----------------------+---------------+
|2025-01-01 00:00:00|744           |0                   |0           |0             |0                      |0              |
|2025-02-01 00:00:00|672           |0                   |0           |0             |0                      |0              |
|2025-03-01 00:00:00|744           |0                   |0           |0             |0                      |0              |
|2025-04-01 00:00:00|720           |684                 |684         |116           |115                    |12             |
|2025-05-01 00:00:00|744           |709                 |709         |57            |52                     |10       

### 10.4a How much does the duration rule matter?

We compare the selected rule's minimum duration with two hours, keeping the same Silver readings and cutoff. The primary rule uses three hours and 0.045 ppm. An alternative-rule Analyse run uses its selected cutoff for both durations.

This shows how many shorter periods the selected minimum leaves out. It does not change the rule or catalogue used for publication.

In [51]:
# Select the first row of each run so each run is counted once.
# The elevated table also contains runs too short for the published catalogue.
runs = elevated.where(F.col("hour_start_utc") == F.col("period_start_utc"))
# Compare the selected primary duration with two hours while keeping the same input and cutoff.
# This display does not replace members or catalogue, so it does not change the published rule.
for minimum in (gold_rules["minimum_hours"], 2):
    totals = runs.where(F.col("run_hours") >= minimum).agg(
        F.count("*").alias("periods"),
        F.coalesce(F.sum("run_hours"), F.lit(0)).alias("member_hours"),
    ).first()
    print(f"Minimum {minimum} hours: {totals.periods} periods, {totals.member_hours} member hours")

Minimum 3 hours: 74 periods, 483 member hours
Minimum 2 hours: 89 periods, 513 member hours


### 10.4b Publish and reload Gold

We save the period catalogue and monthly coverage together. The release manifest records the files, the hourly Silver input and the selected ozone rule. We reopen the release and check both tables before writing the completion report.

In [52]:
# Publish the catalogue and monthly coverage with the selected rule and exact Silver parent.
from gold_io import publish_gold, load_gold, write_gold_completion

gold_release = publish_gold(
    catalogue=catalogue, monthly=months,
    rules=gold_rules, parent=gold_parent, parent_path=gold_parent_path,
    parent_sha256=gold_parent_sha, context=silver_context, rule_path=rule_path,
)

# Reopen the publication and verify its manifest, parent checksum and stored tables.
released_manifest, released_tables = load_gold(
    gold_release["manifest_path"],
    expected_sha256=gold_release["manifest_sha256"],
    expected_parent_sha256=gold_parent_sha,
    context=silver_context, spark=spark,
)

# Write the success receipt only after the Gold release has reloaded.
gold_completion_path = write_gold_completion(silver_context, released_manifest, gold_release, write_json)
# Release the cached elevated hours now that all Gold outputs are saved.
elevated.unpersist()

# Show the release identity, parent and saved row counts for later verification.
print("Gold publication:", released_manifest["publication_id"])
print("Gold manifest SHA-256:", gold_release["manifest_sha256"])
print("Hourly Silver parent:", released_manifest["parent_hourly"]["publication_id"])
print("Reloaded Gold tables:", {k: v["records"] for k, v in released_manifest["datasets"].items()})
print("Completion report:", gold_completion_path)

Gold publication: aa53e8b28e14304139263faf5076434286900b5320ddf0e1d3ca3a7ec2719a0a
Gold manifest SHA-256: cc5ee659fe695ae982f9866d03c04e00ea5ca244e94bb94ae2d3ea7fd1680220
Hourly Silver parent: c22c1b3b6f61e6896a09e5845469903776890bb2a63b3af96f8c519c73ad63b9
Reloaded Gold tables: {'period_catalogue': 74, 'monthly_coverage': 12}
Completion report: /home/ubuntu/dat535-final-project-data/dev/runs/bc50a11d476d4e0083d429448023dcef/reports/gold_completion.json


### Georgia: verify reuse of the Silver pipeline

We use the South DeKalb monitor and Atlanta station to check that the same cleaning and hourly matching functions work for a second region. The pairing remains `checked_unpublished` pending review of the EPA coordinate datum.

To reproduce the check after the Alabama run, call this function in the same kernel:

```python
check_georgia(downloads, georgia_profile, reference_raw_tables, reference_sources, silver_context)
```

Running this cell only defines `check_georgia`. Calling it returns an in-memory summary and prints the results. It does not publish a Georgia source or hourly release, record acceptance, or run Gold.

In [53]:
# Define a separately called Georgia reuse check; this does not publish Georgia Silver or Gold.
def check_georgia(downloads, georgia_profile, reference_raw_tables, reference_sources, silver_context):
    """Run the shared Silver transformations for Georgia and return an unpublished check summary."""
    def portability_source(**identity):
        """Return the one selected Bronze source matching the requested identity."""
        # Find exactly one selected source matching the supplied provider, dataset, year and station.
        matches = [
            item for item in downloads
            if all(item["request"].get(key) == value for key, value in identity.items())
        ]
        if len(matches) != 1:
            raise ValueError(f"Expected one selected Bronze source: {identity}")
        return matches[0]


    # Run the same ozone and temperature processors using Georgia's profile.
    georgia_portability = None
    ga_ozone = process_ozone_source(
        portability_source(
            source="epa", dataset="hourly_ozone", year=georgia_profile["year"]
        ),
        georgia_profile, silver_context.environment_root,
    )
    ga_temperature = process_temperature_source(
        portability_source(
            source="noaa", dataset="hourly_weather", year=georgia_profile["year"],
            station_id=georgia_profile["weather_mapping"]["noaa_station_id"],
        ),
        georgia_profile, silver_context.environment_root,
    )
    # Build the Georgia identifiers used to select its three reference records.
    ga_monitor = georgia_profile["ozone_monitor"]
    ga_site_keys = {
        "State Code": ga_monitor["state_code"],
        "County Code": ga_monitor["county_code"],
        "Site Number": ga_monitor["site_number"],
    }
    ga_reference_keys = {
        "epa_sites": ga_site_keys,
        "epa_monitors": {
            **ga_site_keys,
            "Parameter Code": ga_monitor["parameter_code"],
            "POC": ga_monitor["poc"],
        },
        "noaa_stations": {
            "GHCN_ID": georgia_profile["weather_mapping"]["noaa_station_id"],
        },
    }
    # Collect the cleaned observations and reference tables needed for hourly matching.
    ga_tables = {
        "ozone_observations": ga_ozone["observations"],
        "temperature_reports": ga_temperature["observations"],
    }
    # Filter the original reference tables, not the rows already selected for Alabama.
    for name, keys in ga_reference_keys.items():
        selected = reference_raw_tables[name]
        for column, value in keys.items():
            selected = selected.filter(F.col(column) == F.lit(value))
        ga_tables[name] = process_reference_records(
            selected, reference_sources[name], name, silver_context.environment_root
        )["reference"]

    # Track only caches created here so the function releases its own cached tables.
    ga_cached = []
    try:
        for table in ga_tables.values():
            if not table.is_cached:
                table.cache()
                ga_cached.append(table)
            # Materialise the cached input before reusing it for the mapping and hourly checks.
            table.count()

        # Check the cleaned inputs and build a mapping without claiming a published source parent.
        ga_input_counts = inspect_hourly_inputs(ga_tables, georgia_profile)
        ga_mapping, ga_mapping_evidence = build_yearly_mapping(
            ga_tables, georgia_profile, source_publication_id=None
        )
        # Select one report per hour and keep ozone readings even when weather is missing.
        ga_hourly_temperature = select_hourly_temperature(
            ga_tables["temperature_reports"],
            policy_version=georgia_profile["temperature_policy"]["version"],
        )
        ga_enriched, ga_enrichment_checks = enrich_hourly_ozone(
            ga_tables["ozone_observations"], ga_hourly_temperature, ga_mapping
        )
        # Count hourly weather and represented reports, then reconcile the match with the cleaned inputs.
        ga_weather_counts = ga_hourly_temperature.agg(
            F.count("*").alias("hours"),
            F.coalesce(F.sum("qualifying_report_count"), F.lit(0)).alias("reports"),
        ).first().asDict()
        if (
            ga_weather_counts["hours"]
            != ga_input_counts["temperature_reports"]["clock_hours"]
            or ga_weather_counts["reports"]
            != ga_temperature["summary"]["after"]["records"]
            or ga_enrichment_checks["enriched_records"]
            != ga_ozone["summary"]["after"]["records"]
            or ga_enrichment_checks["matched_temperature"] == 0
        ):
            raise ValueError(
                "Georgia portability counts failed; inspect the source audits."
            )

        # Build a checked_unpublished summary with source versions, counts and mapping review flags.
        georgia_portability = {
            "status": "checked_unpublished",
            "profile": json.loads(json.dumps(georgia_profile)),
            "source_silver_publication_id": None,
            "source_versions": {
                "ozone": ga_ozone["summary"]["source_version_sha256"],
                "temperature": ga_temperature["summary"]["source_version_sha256"],
            },
            "source_dispositions": {
                "ozone": ga_ozone["summary"]["dispositions"],
                "temperature": ga_temperature["summary"]["dispositions"],
            },
            "inputs": ga_input_counts,
            "hourly_temperature": ga_weather_counts,
            "enrichment": ga_enrichment_checks,
            "mapping_evidence": ga_mapping_evidence,
        }
        print("Georgia portability:", georgia_portability["status"])
        print("Ozone dispositions:",
              georgia_portability["source_dispositions"]["ozone"])
        print("Temperature dispositions:",
              georgia_portability["source_dispositions"]["temperature"])
        print("Hourly temperature:", ga_weather_counts)
        print("Enrichment:", ga_enrichment_checks)
        print("Mapping review flags:", ga_mapping_evidence["review_flags"])

        # Optionally break down temperature audit outcomes by NOAA metadata codes.
        if show_diagnostics:
            ga_temperature["audit"].groupBy(
                F.col("raw_fields")["temperature_Source_Code"].alias("source_code"),
                F.col("raw_fields")["temperature_Quality_Code"].alias("quality_code"),
                F.col("raw_fields")["temperature_Report_Type"].alias("report_type"),
                F.col("raw_fields")["temperature_Measurement_Code"].alias("measurement_code"),
                "disposition",
            ).agg(F.sum("record_count").alias("source_records")).orderBy(
                F.desc("source_records")
            ).show(100, truncate=False)
    # Release caches even if a check fails.
    finally:
        for table in ga_cached:
            table.unpersist(blocking=True)
    # Return the check summary to the caller; this function does not write a publication manifest.
    return georgia_portability

### Spark performance: baseline for hourly temperature matching

We time temperature selection and the join to ozone three times using the same Alabama inputs. Each timing includes collecting the result. Spark's SQL cache is cleared between runs, and we check that every run gives the same rows. The report stores the timings and execution plans for comparison with a later implementation.

To reproduce the baseline after the Alabama run:

```python
benchmark_hourly(hourly_tables, hourly_manifest, prepared_silver, silver_context)
```

Running this cell only defines `benchmark_hourly`. Call it after the full Alabama preparation when you want the baseline report. Clearing Spark SQL caches does not clear the operating system cache or restart the JVM. These timings provide a repeatable baseline on this VM, not evidence of multi-machine scaling or an optimisation speedup.

In [54]:
# Define a separately called timing baseline for hourly temperature selection and ozone matching.
def benchmark_hourly(hourly_tables, hourly_manifest, prepared_silver, silver_context):
    """Time three equivalent hourly preparations and save a baseline report."""
    from contextlib import redirect_stdout
    from io import StringIO


    def hourly_result_probe(table):
        """Represent every output row as JSON, including null fields, for full-row comparison."""
        return table.select(F.to_json(
            F.struct(*[F.col(name) for name in table.columns]),
            options={"ignoreNullFields": "false"},
        ).alias("record_json"))


    def hourly_result_digest(rows):
        """Hash sorted row text, retaining duplicate rows but ignoring their arrival order."""
        return hashlib.sha256(
            "\n".join(sorted(row["record_json"] for row in rows)).encode("utf-8")
        ).hexdigest()


    # Record the expected schema and full-row digest before starting the timed repetitions.
    baseline_expected_table = prepared_silver["tables"]["ozone_enriched"]
    baseline_expected_schema = baseline_expected_table.schema.json()
    baseline_expected_digest = hourly_result_digest(
        hourly_result_probe(baseline_expected_table).collect()
    )
    baseline_runs = []
    baseline_plans = []

    # Repeat three times with Spark SQL caches cleared. OS caches and the running JVM remain.
    for repetition in range(1, 4):
        spark.catalog.clearCache()
        started = time.perf_counter()

        # Rebuild hourly temperature and ozone matching from the same source Silver inputs.
        benchmark_weather = select_hourly_temperature(
            hourly_tables["temperature_reports"],
            policy_version=prepared_silver["rules"]["temperature_policy_version"],
        )
        benchmark_enriched, benchmark_checks = enrich_hourly_ozone(
            hourly_tables["ozone_observations"],
            benchmark_weather,
            prepared_silver["tables"]["yearly_mapping"],
        )
        benchmark_probe = hourly_result_probe(benchmark_enriched)
        # collect triggers the Spark work and transfers all result rows to Python.
        # This is suitable for our small regional output, not a general large-data benchmark.
        benchmark_rows = benchmark_probe.collect()
        elapsed = time.perf_counter() - started

        # After timing, require the schema and all row contents to match the prepared output.
        if (
            benchmark_enriched.schema.json() != baseline_expected_schema
            or hourly_result_digest(benchmark_rows) != baseline_expected_digest
        ):
            raise ValueError(
                "Benchmark output differs from the prepared ozone table."
            )

        # Capture Spark's execution plan alongside the elapsed time for later comparison.
        plan_buffer = StringIO()
        with redirect_stdout(plan_buffer):
            benchmark_probe.explain(mode="formatted")
        baseline_plans.append(plan_buffer.getvalue())
        baseline_runs.append({
            "repetition": repetition,
            "seconds": elapsed,
            "rows": len(benchmark_rows),
        })
        print(
            f"Run {repetition}: {elapsed:.3f} seconds; "
            f"{len(benchmark_rows)} rows"
        )

    # Read the saved code identity so the baseline can be traced to its implementation.
    saved_pipeline = json.loads(
        (project_dir / "notebooks" / "pipeline.ipynb").read_text(encoding="utf-8")
    )
    # Record input sizes, runtime settings, cache conditions, timings and execution plans.
    hourly_performance_baseline = {
        "status": "recorded",
        "scope": (
            "hourly temperature selection, ozone enrichment, "
            "checks and full JSON collection"
        ),
        "source_silver_publication_id": hourly_manifest["publication_id"],
        "saved_notebook_code_sha256": notebook_code_sha256(saved_pipeline),
        "source_files_sha256": saved_code["source_files_sha256"],
        "profile_sha256": silver_json_sha256(prepared_silver["profile"]),
        "rules": prepared_silver["rules"],
        "spark_version": spark.version,
        "master": spark.sparkContext.master,
        "spark_application_id": spark.sparkContext.applicationId,
        "spark_ui_url": spark.sparkContext.uiWebUrl,
        "runtime_settings": {
            name: spark.sparkContext.getConf().get(name, None)
            for name in (
                "spark.executor.cores", "spark.executor.memory",
                "spark.driver.memory", "spark.cores.max",
            )
        },
        "cache_condition": (
            "Spark SQL caches cleared before each run; "
            "OS cache unmanaged; JVM already running"
        ),
        "input_volumes": {
            name: {
                "records": hourly_manifest["datasets"][name]["row_count"],
                "stored_parquet_bytes": hourly_manifest["datasets"][name]["bytes"],
            }
            for name in ("ozone_observations", "temperature_reports")
        },
        "sql_settings": {
            name: spark.conf.get(name)
            for name in (
                "spark.sql.session.timeZone",
                "spark.sql.shuffle.partitions",
                "spark.sql.adaptive.enabled",
                "spark.sql.autoBroadcastJoinThreshold",
            )
        },
        "runs": baseline_runs,
        "median_seconds": sorted(run["seconds"] for run in baseline_runs)[1],
        "result_sha256": baseline_expected_digest,
        "schema_json": baseline_expected_schema,
        "physical_plans": baseline_plans,
    }
    # Save a baseline report for comparison with a later measured alternative.
    hourly_baseline_path = (
        silver_context.reports_dir / "hourly_performance_baseline.json"
    )
    write_json(hourly_baseline_path, hourly_performance_baseline)

    print(
        "Median seconds:",
        round(hourly_performance_baseline["median_seconds"], 3)
    )
    print("All three runs reproduced the complete prepared output.")
    print("Baseline report:", hourly_baseline_path)
    # Return the report path; the three repetitions alone do not demonstrate a speedup.
    return hourly_baseline_path